# COMPAS race-binary CMAB fairness

This notebook runs the COMPAS experiments with binary race as the sensitive attribute for three contextual bandit policy families: LinUCB, Linear Thompson Sampling, and EXP4.

The objective is to compare each standard policy with its fairness-aware variant under two preprocessing settings:

- **Uniform**: the training stream or update process is used without group-label reweighting;
- **Reweighting**: group-label strata are reweighted to reduce imbalance across sensitive-group and label combinations.

The notebook evaluates three levels of fairness intervention:

- **Pre-processing**: uniform processing versus group-label reweighting;
- **In-processing**: standard policy versus demographic-parity-aware policy;
- **Post-processing**: group-specific threshold calibration on held-out data.

The binary action is interpreted as the predicted recidivism class. The reward is equal to 1 when the selected action matches the observed label and 0 otherwise. Therefore, average reward is equivalent to accuracy in this offline classification-derived bandit setting.

The main utility metrics are average reward and cumulative prediction error. Fairness is assessed using Demographic Parity Gap, Equalized Odds Gap, and UtilityGap. Temporal figures report the evolution of these metrics over the learning horizon.

The COMPAS notebook uses the generic classification-derived CMAB runner. This runner is shared with the Adult notebook because both datasets are offline binary classification benchmarks transformed into contextual bandit replay environments.

## Imports and configuration

This section loads the project modules, defines the run mode, output directories, random seeds, horizon, policy-family hyperparameters, preprocessing settings, and post-processing settings.

For reproducibility, the notebook can either regenerate the full benchmark or reload cached outputs.

In [ ]:
from __future__ import annotations

from dataclasses import replace

import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.stats import wilcoxon, t
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fair_bandits.config import build_config
from fair_bandits.data.compas import (
    clean_compas_minimal,
    ensure_compas_race_binary,
    load_compas,
    prepare_compas_contextual,
)

from fair_bandits.metrics import (
    normalize_metric_columns,
)

from fair_bandits.experiments import (
    ClassificationBanditParams,
    load_classification_family_outputs,
    run_classification_family_benchmark,
    train_classification_expert_pool,
    tune_random_forest_hyperparameters,
    run_random_forest_benchmark,
)

from fair_bandits.plots import (
    plot_compas_family_figure_set,
    plot_compas_final_dot_plots,
    plot_compas_postprocessing_over_horizon,
    plot_compas_linucb_inprocessing_average_reward,
    plot_compas_linucb_postprocessing_average_reward,
)

from fair_bandits.io import export_latex_table, fmt_mean_sd

from fair_bandits.plots import plot_real_dataset_tradeoff_set

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


In [ ]:
RUN_MODE = "full"
CFG = build_config("dev" if RUN_MODE == "dev" else "full")

RESULTS_ROOT = CFG.results_dir
OUTPUT_ROOT = RESULTS_ROOT / "compas_race_binary_fairness"
RUN_DIR = OUTPUT_ROOT / RUN_MODE
FIG_DIR = OUTPUT_ROOT / "final_figures"
TABLE_DIR = OUTPUT_ROOT / "overleaf_tables"

for directory in [RUN_DIR, FIG_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PREPROCESSINGS = ["uniform", "reweigh_group_label"]
POLICY_FAMILIES = {
    "linucb": {
        "label": "LinUCB",
        "policies": ["LinUCB", "FairLinUCB"],
        "postprocessed_policy": "FairLinUCB+PP",
    },
    "linear_ts": {
        "label": "Linear Thompson Sampling",
        "policies": ["LinTS", "FairLinTS"],
        "postprocessed_policy": "FairLinTS+PP",
    },
    "exp4": {
        "label": "EXP4",
        "policies": ["EXP4", "FairEXP4"],
        "postprocessed_policy": "FairEXP4+PP",
    },
}

POLICY_LABELS = {
    "LinUCB": "LinUCB",
    "FairLinUCB": "FairLinUCB",
    "FairLinUCB+PP": "FairLinUCB+PP",
    "LinTS": "LinTS",
    "FairLinTS": "FairLinTS",
    "FairLinTS+PP": "FairLinTS+PP",
    "EXP4": "EXP4",
    "FairEXP4": "FairEXP4",
    "FairEXP4+PP": "FairEXP4+PP",
}

PREPROCESSING_LABELS = {
    "uniform": "Uniform",
    "reweigh_group_label": "Reweighting",
}

POSITIVE_CLASS = 1

if RUN_MODE == "dev":
    N_SEEDS = 2
    T_MAX = 1500
    LOG_EVERY = 25
    N_EXPERTS = 8
    EXPERT_BOOTSTRAP_SIZE = 4000
else:
    N_SEEDS = 50
    T_MAX = 30000
    LOG_EVERY = 50
    N_EXPERTS = 20
    EXPERT_BOOTSTRAP_SIZE = 8000

SEEDS = list(range(N_SEEDS))
TEST_SIZE = 0.20
CALIBRATION_SIZE_WITHIN_REMAINING = 0.20
EXP4_GAMMA = 0.07
EXP4_ETA = None
DP_TAU = 0.02
DP_LAMBDA = 2.0
BETA_SMOOTH = 1.0
MIN_GROUP_COUNT = 20
MAX_ACCURACY_DROP = 0.01
THRESHOLD_GRID_SIZE = 31
RUN_BENCHMARK = True
FORCE_RERUN = False
RANDOM_STATE = 42
ALPHA_LINUCB = 1.5
LAMBDA_RIDGE = 10.0
TS_V = 0.25

print("RUN_MODE:", RUN_MODE)
print("RUN_DIR:", RUN_DIR)
print("N_SEEDS:", N_SEEDS)
print("T_MAX:", T_MAX)
print("LOG_EVERY:", LOG_EVERY)
print("N_EXPERTS:", N_EXPERTS)


## Load and prepare COMPAS binary-race data

The COMPAS dataset is loaded and prepared with binary race as the sensitive attribute.

The original supervised dataset is transformed into an offline contextual bandit environment. At each round, the learner observes one individual context, selects a binary action, and receives reward 1 if the selected action matches the observed label.

In [ ]:
df_raw = load_compas(cache_dir=CFG.cache_dir)
df_raw = clean_compas_minimal(df_raw)
df_raw = ensure_compas_race_binary(df_raw)

X, y, group, feature_names, meta = prepare_compas_contextual(
    df_raw,
    sensitive_col="race_binary",
)

compas_df = df_raw.copy()
X_df = pd.DataFrame(X, columns=feature_names)
y_series = pd.Series(y, name="target")
group_series = pd.Series(group, name="race_binary")

print("Shape after cleaning:", df_raw.shape)
print("Prepared X shape:", X.shape)
display(df_raw.head())

## Split, standardize, and build expert advice

This section creates train, calibration, and test splits, standardizes the feature matrix, and trains the supervised expert pool required by EXP4.

The expert pool provides action-advice matrices used by EXP4 and FairEXP4 during the contextual bandit replay.

In [ ]:
strata = (
    group_series.astype(str)
    + "||"
    + y_series.astype(str)
)

(
    X_train_df,
    X_test_df,
    y_train_s,
    y_test_s,
    g_train_s,
    g_test_s,
) = train_test_split(
    X_df,
    y_series,
    group_series,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=strata,
)

strata_remaining = (
    g_train_s.astype(str)
    + "||"
    + y_train_s.astype(str)
)

(
    X_train_df,
    X_cal_df,
    y_train_s,
    y_cal_s,
    g_train_s,
    g_cal_s,
) = train_test_split(
    X_train_df,
    y_train_s,
    g_train_s,
    test_size=CALIBRATION_SIZE_WITHIN_REMAINING,
    random_state=43,
    stratify=strata_remaining,
)

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train_df
)

X_cal = scaler.transform(
    X_cal_df
)

X_test = scaler.transform(
    X_test_df
)

y_train = y_train_s.to_numpy(
    dtype=int
)

y_cal = y_cal_s.to_numpy(
    dtype=int
)

y_test = y_test_s.to_numpy(
    dtype=int
)

g_train = g_train_s.astype(str).to_numpy()
g_cal = g_cal_s.astype(str).to_numpy()
g_test = g_test_s.astype(str).to_numpy()

print(
    "Train:",
    X_train.shape,
    "Calibration:",
    X_cal.shape,
    "Test:",
    X_test.shape,
)

print(
    "Groups:",
    sorted(
        np.unique(
            g_train
        )
    ),
)

In [ ]:
# ============================================================
# COMPAS: supervised Random Forest baseline
# ============================================================

RF_PATH = RUN_DIR / "random_forest_endpoint.csv"
RF_TUNING_PATH = RUN_DIR / "random_forest_tuning.csv"

RUN_RF_BENCHMARK = True
FORCE_RF_RERUN = False


if RF_PATH.exists() and not FORCE_RF_RERUN:

    print("Loading cached Random Forest results:")
    print(RF_PATH)

    compas_rf_df = normalize_metric_columns(
        pd.read_csv(RF_PATH)
    )

else:

    # --------------------------------------------------------
    # Hyperparameter tuning on TRAINING DATA ONLY
    #
    # Tuning is performed once on the ordinary/unweighted
    # training distribution. The selected hyperparameters are
    # then kept identical for Uniform and Reweighting.
    # --------------------------------------------------------

    print("Tuning Random Forest hyperparameters...")

    rf_params, rf_tuning_df = (
        tune_random_forest_hyperparameters(
            X_train=X_train,
            y_train=y_train,
            seed=42,
            n_estimators=500,
            cv_folds=3,
            n_jobs=-1,
        )
    )

    print("\nSelected RF parameters:")
    print(rf_params)

    display(
        rf_tuning_df.head(10)
    )

    rf_tuning_df.to_csv(
        RF_TUNING_PATH,
        index=False,
    )

    print(
        "Saved tuning results:",
        RF_TUNING_PATH,
    )

    # --------------------------------------------------------
    # 50 seeds × 2 preprocessing conditions
    # Evaluation is performed ONLY on the held-out test set
    # --------------------------------------------------------

    print("\nRunning Random Forest benchmark...")

    compas_rf_df = run_random_forest_benchmark(
        X_train=X_train,
        y_train=y_train,
        g_train=g_train,
        X_test=X_test,
        y_test=y_test,
        g_test=g_test,
        seeds=SEEDS,
        preprocessings=PREPROCESSINGS,
        params=rf_params,
        output_path=RF_PATH,
    )

    print(
        "Saved Random Forest results:",
        RF_PATH,
    )


print("\nRandom Forest output shape:")
print(compas_rf_df.shape)

display(
    compas_rf_df.groupby(
        ["policy", "preprocessing"]
    )[
        [
            "average_reward",
            "DP_gap",
            "EO_gap",
            "TPR_gap",
            "FPR_gap",
            "UtilityGap",
        ]
    ].agg(
        ["mean", "std"]
    )
)

## Shared bandit parameters

This cell builds the common parameter object used by the LinUCB, Linear Thompson Sampling, and EXP4 family benchmarks. The feature dimension is inferred after preprocessing from the standardized COMPAS design matrix.

In [ ]:
params = ClassificationBanditParams(
    d=X_train.shape[1],
    t_max=T_MAX,
    log_every=LOG_EVERY,
    alpha_linucb=ALPHA_LINUCB,
    ts_v=TS_V,
    lambda_ridge=LAMBDA_RIDGE,
    exp4_gamma=EXP4_GAMMA,
    exp4_eta=EXP4_ETA,
    n_experts=N_EXPERTS,
    dp_tau=DP_TAU,
    dp_lambda=DP_LAMBDA,
    beta_smooth=BETA_SMOOTH,
    min_group_count=MIN_GROUP_COUNT,
    max_accuracy_drop=MAX_ACCURACY_DROP,
    threshold_grid_size=THRESHOLD_GRID_SIZE,
)

print(params)

## Train expert pool and cache expert advice

This section trains the supervised expert pool used by EXP4 and FairEXP4, then precomputes the expert advice matrices for the train, calibration, and test splits.

In [ ]:
print("Training expert pool...")

expert_pool = train_classification_expert_pool(
    X_train,
    y_train,
    n_experts=N_EXPERTS,
    bootstrap_size=EXPERT_BOOTSTRAP_SIZE,
    seed=2026,
)

print("Computing cached expert advice...")

ADVICE_TRAIN = expert_pool.predict_advice(X_train)
ADVICE_CAL = expert_pool.predict_advice(X_cal)
ADVICE_TEST = expert_pool.predict_advice(X_test)

print("ADVICE_TRAIN:", ADVICE_TRAIN.shape)
print("ADVICE_CAL:  ", ADVICE_CAL.shape)
print("ADVICE_TEST: ", ADVICE_TEST.shape)


## Preflight

The preflight runs a short sanity check before launching or loading the full benchmark.

It verifies that each policy family can run a short trajectory, update its internal state, and produce the expected endpoint and post-processing outputs.

In [ ]:
RUN_PREFLIGHT = True

if RUN_PREFLIGHT:
    print("Running family preflight...")

    preflight_params = ClassificationBanditParams(
        d=X_train.shape[1],
        t_max=min(500, T_MAX),
        log_every=100,
        alpha_linucb=ALPHA_LINUCB,
        ts_v=TS_V,
        lambda_ridge=LAMBDA_RIDGE,
        exp4_gamma=EXP4_GAMMA,
        exp4_eta=EXP4_ETA,
        n_experts=N_EXPERTS,
        dp_tau=DP_TAU,
        dp_lambda=DP_LAMBDA,
        beta_smooth=BETA_SMOOTH,
        min_group_count=MIN_GROUP_COUNT,
        max_accuracy_drop=MAX_ACCURACY_DROP,
        threshold_grid_size=THRESHOLD_GRID_SIZE,
    )

    for family, family_config in POLICY_FAMILIES.items():
        print("\nPreflight family:", family)

        _ = run_classification_family_benchmark(
            family=family,
            run_dir=RUN_DIR / "_preflight" / family,
            X_train=X_train,
            y_train=y_train,
            g_train=g_train,
            X_cal=X_cal,
            y_cal=y_cal,
            g_cal=g_cal,
            X_test=X_test,
            y_test=y_test,
            g_test=g_test,
            advice_train=ADVICE_TRAIN if family == "exp4" else None,
            advice_cal=ADVICE_CAL if family == "exp4" else None,
            advice_test=ADVICE_TEST if family == "exp4" else None,
            preprocessings=["uniform"],
            policies=family_config["policies"],
            seeds=[0],
            params=preflight_params,
            force_rerun=True,
        )

    print("Family preflight completed.")

## Run or load COMPAS policy-family benchmarks

This section either runs the full COMPAS binary-race benchmarks or reloads cached outputs.

The benchmark evaluates LinUCB, Linear Thompson Sampling, and EXP4 families across both preprocessing settings, all random seeds, and the full learning horizon. The outputs are concatenated into temporal trajectories, endpoint summaries, post-processing results, and post-processing parameter tables.

In [ ]:
family_outputs = {}

for family, family_config in POLICY_FAMILIES.items():
    print("\n" + "=" * 80)
    print("Family:", family)

    family_run_dir = RUN_DIR / family

    advice_train = ADVICE_TRAIN if family == "exp4" else None
    advice_cal = ADVICE_CAL if family == "exp4" else None
    advice_test = ADVICE_TEST if family == "exp4" else None

    if RUN_BENCHMARK:
        family_outputs[family] = run_classification_family_benchmark(
            family=family,
            run_dir=family_run_dir,
            X_train=X_train,
            y_train=y_train,
            g_train=g_train,
            X_cal=X_cal,
            y_cal=y_cal,
            g_cal=g_cal,
            X_test=X_test,
            y_test=y_test,
            g_test=g_test,
            advice_train=advice_train,
            advice_cal=advice_cal,
            advice_test=advice_test,
            preprocessings=PREPROCESSINGS,
            policies=family_config["policies"],
            seeds=SEEDS,
            params=params,
            force_rerun=FORCE_RERUN,
        )

    else:
        family_outputs[family] = load_classification_family_outputs(
            family=family,
            run_dir=family_run_dir,
        )

temporal_df = pd.concat(
    [outputs[0] for outputs in family_outputs.values()],
    ignore_index=True,
)

endpoint_df = pd.concat(
    [outputs[1] for outputs in family_outputs.values()],
    ignore_index=True,
)

postproc_df = pd.concat(
    [outputs[2] for outputs in family_outputs.values()],
    ignore_index=True,
)

parameters_df = pd.concat(
    [outputs[3] for outputs in family_outputs.values()],
    ignore_index=True,
)

print("\nCombined outputs")
print("temporal_df:  ", temporal_df.shape)
print("endpoint_df:  ", endpoint_df.shape)
print("postproc_df:  ", postproc_df.shape)
print("parameters_df:", parameters_df.shape)

display(
    endpoint_df.groupby(
        ["family", "policy", "preprocessing"]
    )["seed"].nunique()
)

## Longitudinal post-processing benchmark

This optional section evaluates held-out post-processing at several training horizons.

For each policy family, the benchmark is rerun with a shorter training horizon, and the post-processing step is then applied on the held-out calibration and test sets. The resulting table is used only to generate the post-processing-over-horizon curves. The main endpoint summaries and significance tests remain based on the final horizon.

In [ ]:
POSTPROC_HORIZONS = [
    500,
    1000,
    2000,
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
]

POSTPROC_HORIZON_DIR = RUN_DIR / "postprocessing_horizons"
POSTPROC_HORIZON_PATH = (
    POSTPROC_HORIZON_DIR
    / "compas_race_binary_postprocessing_over_horizon.csv"
)

RUN_POSTPROC_HORIZON_BENCHMARK = False

POSTPROC_HORIZON_FAMILIES = [
    "linucb",
    "linear_ts",
    "exp4",
]

if RUN_POSTPROC_HORIZON_BENCHMARK:
    horizon_rows = []

    for horizon in POSTPROC_HORIZONS:
        print("\nPost-processing horizon:", horizon)

        horizon_params = replace(
            params,
            t_max=int(horizon),
        )

        for family in POSTPROC_HORIZON_FAMILIES:
            family_config = POLICY_FAMILIES[family]

            print("  Family:", family)

            family_horizon_dir = (
                POSTPROC_HORIZON_DIR
                / family
                / f"horizon_{horizon}"
            )

            _, _, family_postproc_horizon_df, _ = run_classification_family_benchmark(
                family=family,
                run_dir=family_horizon_dir,
                X_train=X_train,
                y_train=y_train,
                g_train=g_train,
                X_cal=X_cal,
                y_cal=y_cal,
                g_cal=g_cal,
                X_test=X_test,
                y_test=y_test,
                g_test=g_test,
                advice_train=ADVICE_TRAIN if family == "exp4" else None,
                advice_cal=ADVICE_CAL if family == "exp4" else None,
                advice_test=ADVICE_TEST if family == "exp4" else None,
                preprocessings=PREPROCESSINGS,
                policies=family_config["policies"],
                seeds=SEEDS,
                params=horizon_params,
                force_rerun=FORCE_RERUN,
            )

            family_postproc_horizon_df = family_postproc_horizon_df.copy()
            family_postproc_horizon_df["horizon"] = int(horizon)
            family_postproc_horizon_df["t"] = int(horizon)

            horizon_rows.append(family_postproc_horizon_df)

    if not horizon_rows:
        raise RuntimeError(
            "No longitudinal post-processing rows were generated."
        )

    postproc_horizon_df = normalize_metric_columns(
        pd.concat(
            horizon_rows,
            ignore_index=True,
        )
    )

    POSTPROC_HORIZON_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    postproc_horizon_df.to_csv(
        POSTPROC_HORIZON_PATH,
        index=False,
    )

    print("Saved:", POSTPROC_HORIZON_PATH)

else:
    if POSTPROC_HORIZON_PATH.exists():
        postproc_horizon_df = normalize_metric_columns(
            pd.read_csv(POSTPROC_HORIZON_PATH)
        )

        print("Loaded:", POSTPROC_HORIZON_PATH)

    else:
        postproc_horizon_df = pd.DataFrame()

        print(
            "No longitudinal post-processing file found. "
            "Post-processing-over-horizon figures will be skipped."
        )

print("postproc_horizon_df:", postproc_horizon_df.shape)

if not postproc_horizon_df.empty:
    display(
        postproc_horizon_df.groupby(
            ["family", "policy", "preprocessing"]
        )["horizon"].nunique()
    )

## Figures

This section generates the standard COMPAS binary-race figure set for each policy family.

For each family, the notebook produces:

- a preprocessing comparison for Demographic Parity Gap and Equalized Odds Gap;
- an in-processing comparison under uniform preprocessing;
- temporal utility figures for average reward, cumulative prediction error, and UtilityGap;
- final predictive and fairness dot plots;
- optional post-processing-over-horizon curves when the longitudinal post-processing file is available.

The visual convention is kept consistent across families: baseline policies are blue, fairness-aware policies are red, post-processed policies are green, uniform preprocessing is shown with solid lines, and reweighting with dotted lines.

Temporal bands indicate pointwise 95% confidence intervals for the mean trajectory across seeds.

In [ ]:
print("Temporal:", temporal_df.shape if not temporal_df.empty else "MISSING/EMPTY")
print("Endpoint:", endpoint_df.shape if not endpoint_df.empty else "MISSING/EMPTY")
print("Postprocessing:", postproc_df.shape if not postproc_df.empty else "MISSING/EMPTY")

In [ ]:
figure_paths = []

for family, family_config in POLICY_FAMILIES.items():
    figure_paths.extend(
        plot_compas_family_figure_set(
            temporal_df=temporal_df,
            family=family,
            family_label=family_config["label"],
            policies=family_config["policies"],
            preprocessings=PREPROCESSINGS,
            fig_dir=FIG_DIR,
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            show=True,
        )
    )

    figure_paths.extend(
        plot_compas_final_dot_plots(
            endpoint_df=endpoint_df,
            postproc_df=postproc_df,
            family=family,
            family_label=family_config["label"],
            policies=family_config["policies"],
            preprocessings=PREPROCESSINGS,
            fig_dir=FIG_DIR,
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            show=True,
        )
    )

    if not postproc_horizon_df.empty:
        baseline_policy, fair_policy = family_config["policies"]

        family_postproc_horizon_df = postproc_horizon_df[
            postproc_horizon_df["family"] == family
        ].copy()

        if not family_postproc_horizon_df.empty:
            figure_paths.append(
                plot_compas_postprocessing_over_horizon(
                    postproc_horizon_df=family_postproc_horizon_df,
                    temporal_df=temporal_df,
                    family=family,
                    family_label=family_config["label"],
                    baseline_policy=baseline_policy,
                    fair_policy=fair_policy,
                    horizons=POSTPROC_HORIZONS,
                    preprocessings=PREPROCESSINGS,
                    fig_dir=FIG_DIR,
                    policy_labels=POLICY_LABELS,
                    preprocessing_labels=PREPROCESSING_LABELS,
                    show=True,
                )
            )
        else:
            print(
                "Skipping post-processing-over-horizon figure for",
                family,
                "because no longitudinal post-processing data are available.",
            )

# Focused LinUCB in-processing average-reward figure
figure_paths.append(
    plot_compas_linucb_inprocessing_average_reward(
        temporal_df=temporal_df,
        fig_dir=FIG_DIR,
        policy_labels=POLICY_LABELS,
        show=True,
    )
)

# Focused LinUCB post-processing average-reward figure
if not postproc_horizon_df.empty:
    figure_paths.append(
        plot_compas_linucb_postprocessing_average_reward(
            postproc_horizon_df=postproc_horizon_df,
            fig_dir=FIG_DIR,
            policy_labels=POLICY_LABELS,
            show=True,
        )
    )

print("Generated figures:", len(figure_paths))

for path in figure_paths:
    print(path)

## Trade-off plots

In [ ]:
COMPAS_TRADEOFF_DIR = FIG_DIR / "tradeoff"

compas_tradeoff_paths = plot_real_dataset_tradeoff_set(
    dataset_label="COMPAS",
    sensitive_label="race binary",
    endpoint_df=endpoint_df,
    postproc_df=postproc_df,
    policy_families=POLICY_FAMILIES,
    preprocessings=PREPROCESSINGS,
    fig_dir=COMPAS_TRADEOFF_DIR,
    file_prefix="compas_race_binary",
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
    confidence=0.95,
    show=True,
)

print("COMPAS trade-off figures:", len(compas_tradeoff_paths))
for path in compas_tradeoff_paths:
    print(path)

## Final summary tables

This section exports compact final summary tables for the COMPAS binary-race experiments.

One table is generated for each policy family: LinUCB, Linear Thompson Sampling, and EXP4. Each table combines the online endpoint results with the held-out post-processed results.

Average reward, cumulative prediction error, Demographic Parity Gap, Equalized Odds Gap, and UtilityGap are reported at the final checkpoint. Values are reported as mean ± standard deviation across seeds.

In [ ]:
all_summary_tables = {}

metrics_for_table = [
    ("average_reward", "Average reward"),
    ("cumulative_prediction_error", "Cumulative prediction error"),
    ("DP_gap", "DP gap"),
    ("EO_gap", "EO gap"),
    ("UtilityGap", "UtilityGap"),
]

preprocessing_order = {
    "uniform": 0,
    "reweigh_group_label": 1,
}

for family, family_config in POLICY_FAMILIES.items():
    online_final = (
        endpoint_df[endpoint_df["family"] == family]
        .sort_values("t")
        .groupby(
            ["seed", "policy", "preprocessing"],
            as_index=False,
        )
        .tail(1)
        .copy()
    )

    postprocessed_policy = family_config["postprocessed_policy"]

    heldout_postproc = (
        postproc_df[
            (postproc_df["family"] == family)
            & (postproc_df["policy"] == postprocessed_policy)
        ]
        .copy()
    )

    performance_source = pd.concat(
        [
            online_final,
            heldout_postproc,
        ],
        ignore_index=True,
    )

    policy_order = {
        family_config["policies"][0]: 0,
        family_config["policies"][1]: 1,
        postprocessed_policy: 2,
    }

    rows = []

    for (policy, preprocessing), group_df in performance_source.groupby(
        ["policy", "preprocessing"]
    ):
        row = {
            "Policy": POLICY_LABELS.get(policy, policy),
            "Preprocessing": PREPROCESSING_LABELS.get(
                preprocessing,
                preprocessing,
            ),
            "Seeds": int(group_df["seed"].nunique()),
        }

        for metric, label in metrics_for_table:
            row[label] = fmt_mean_sd(group_df[metric])

        row["_policy_order"] = policy_order.get(policy, 99)
        row["_preprocessing_order"] = preprocessing_order.get(
            preprocessing,
            99,
        )

        rows.append(row)

    table = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "_policy_order",
                "_preprocessing_order",
            ]
        )
        .drop(
            columns=[
                "_policy_order",
                "_preprocessing_order",
            ]
        )
        .reset_index(drop=True)
    )

    all_summary_tables[family] = table

    display(table)

    csv_path = (
        TABLE_DIR
        / f"compas_race_binary_{family}_performance_summary.csv"
    )

    tex_path = (
        TABLE_DIR
        / f"compas_race_binary_{family}_performance_summary.tex"
    )

    table.to_csv(
        csv_path,
        index=False,
    )

    export_latex_table(
        table,
        tex_path,
        caption=(
            f"Final performance summary for the COMPAS dataset using binary race "
            f"as sensitive attribute and {family_config['label']}, including "
            f"post-processing. Values are reported as mean $\\pm$ standard "
            f"deviation across seeds."
        ),
        label=f"tab:compas_race_binary_{family}_performance_summary",
    )

    print("Saved:", csv_path)
    print("Saved:", tex_path)

## Paired significance tests

This section performs paired Wilcoxon signed-rank tests across seeds.

The comparisons are defined separately for each policy family and cover the three intervention levels:

- preprocessing: baseline policy under uniform preprocessing versus reweighting;
- in-processing: baseline policy versus its fairness-aware variant;
- post-processing: fairness-aware policy before versus after group-specific threshold calibration.

Online comparisons are evaluated at the final online endpoint. Post-processing comparisons are evaluated on the held-out test set. Holm correction is applied for multiple comparisons.


In [ ]:
def safe_wilcoxon(
    baseline,
    intervention,
) -> float:
    baseline = np.asarray(
        baseline,
        dtype=float,
    )

    intervention = np.asarray(
        intervention,
        dtype=float,
    )

    mask = (
        np.isfinite(baseline)
        & np.isfinite(intervention)
    )

    baseline = baseline[mask]
    intervention = intervention[mask]

    if len(baseline) == 0:
        return np.nan

    if np.allclose(
        baseline,
        intervention,
    ):
        return 1.0

    return float(
        wilcoxon(
            baseline,
            intervention,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto",
        ).pvalue
    )


def holm_adjust(
    p_values,
) -> np.ndarray:
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    p_values = np.where(
        np.isfinite(p_values),
        p_values,
        1.0,
    )

    order = np.argsort(p_values)

    adjusted = np.empty(
        len(p_values),
        dtype=float,
    )

    previous = 0.0

    for rank, index in enumerate(order):
        value = (
            len(p_values)
            - rank
        ) * p_values[index]

        value = min(
            max(
                value,
                previous,
            ),
            1.0,
        )

        adjusted[index] = value
        previous = value

    return adjusted


def format_p_value(
    p_value,
) -> str:
    if not np.isfinite(p_value):
        return "--"

    if p_value < 0.001:
        return "<0.001"

    return f"{p_value:.3f}"


def format_significance(
    p_value,
) -> str:
    if not np.isfinite(p_value):
        return ""

    return "*" if p_value < 0.05 else ""


def ensure_cumulative_prediction_error(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    df = dataframe.copy()

    if (
        "cumulative_prediction_error" not in df.columns
        and "cumulative_regret" in df.columns
    ):
        df["cumulative_prediction_error"] = df["cumulative_regret"]

    return df


online_final_stats = (
    ensure_cumulative_prediction_error(endpoint_df)
    .sort_values("t")
    .groupby(
        [
            "family",
            "seed",
            "policy",
            "preprocessing",
        ],
        as_index=False,
    )
    .tail(1)
    .copy()
)

heldout_postproc_stats = ensure_cumulative_prediction_error(
    postproc_df
)


online_metrics = [
    ("DP_gap", "DP gap"),
    ("EO_gap", "EO gap"),
    ("UtilityGap", "UtilityGap"),
    ("average_reward", "Average reward"),
    ("cumulative_prediction_error", "Cumulative prediction error"),
]

heldout_metrics = [
    ("DP_gap", "DP gap"),
    ("EO_gap", "EO gap"),
    ("UtilityGap", "UtilityGap"),
    ("average_reward", "Average reward"),
]


rows = []

for family, family_config in POLICY_FAMILIES.items():
    baseline_policy, fair_policy = family_config["policies"]
    postprocessed_policy = family_config["postprocessed_policy"]

    family_label = family_config["label"]

    comparisons = [
        (
            "Preprocessing effect",
            "online",
            baseline_policy,
            "uniform",
            baseline_policy,
            "reweigh_group_label",
            f"{baseline_policy} | Uniform",
            f"{baseline_policy} | Reweighting",
        ),
        (
            "Preprocessing effect",
            "online",
            fair_policy,
            "uniform",
            fair_policy,
            "reweigh_group_label",
            f"{fair_policy} | Uniform",
            f"{fair_policy} | Reweighting",
        ),
        (
            "In-processing effect",
            "online",
            baseline_policy,
            "uniform",
            fair_policy,
            "uniform",
            f"{baseline_policy} | Uniform",
            f"{fair_policy} | Uniform",
        ),
        (
            "In-processing effect",
            "online",
            baseline_policy,
            "reweigh_group_label",
            fair_policy,
            "reweigh_group_label",
            f"{baseline_policy} | Reweighting",
            f"{fair_policy} | Reweighting",
        ),
        (
            "Post-processing effect",
            "heldout",
            fair_policy,
            "uniform",
            postprocessed_policy,
            "uniform",
            f"{fair_policy} | Uniform",
            f"{postprocessed_policy} | Uniform",
        ),
        (
            "Post-processing effect",
            "heldout",
            fair_policy,
            "reweigh_group_label",
            postprocessed_policy,
            "reweigh_group_label",
            f"{fair_policy} | Reweighting",
            f"{postprocessed_policy} | Reweighting",
        ),
    ]

    for (
        comparison,
        source_name,
        baseline_policy_name,
        baseline_preprocessing,
        intervention_policy_name,
        intervention_preprocessing,
        baseline_label,
        intervention_label,
    ) in comparisons:
        if source_name == "online":
            source_df = online_final_stats[
                online_final_stats["family"] == family
            ].copy()

            metrics = online_metrics

        else:
            source_df = heldout_postproc_stats[
                heldout_postproc_stats["family"] == family
            ].copy()

            metrics = heldout_metrics

        for metric, metric_label in metrics:
            if metric not in source_df.columns:
                print(
                    "Skipping missing metric:",
                    family,
                    metric,
                )
                continue

            baseline = (
                source_df[
                    (
                        source_df["policy"]
                        == baseline_policy_name
                    )
                    & (
                        source_df["preprocessing"]
                        == baseline_preprocessing
                    )
                ][
                    [
                        "seed",
                        metric,
                    ]
                ]
                .rename(
                    columns={
                        metric: "baseline_value",
                    }
                )
            )

            intervention = (
                source_df[
                    (
                        source_df["policy"]
                        == intervention_policy_name
                    )
                    & (
                        source_df["preprocessing"]
                        == intervention_preprocessing
                    )
                ][
                    [
                        "seed",
                        metric,
                    ]
                ]
                .rename(
                    columns={
                        metric: "intervention_value",
                    }
                )
            )

            paired = baseline.merge(
                intervention,
                on="seed",
                how="inner",
                validate="one_to_one",
            )

            if paired.empty:
                print(
                    "Skipping empty paired comparison:",
                    family,
                    comparison,
                    metric_label,
                    baseline_label,
                    intervention_label,
                )
                continue

            raw_p = safe_wilcoxon(
                paired["baseline_value"],
                paired["intervention_value"],
            )

            rows.append(
                {
                    "Family": family_label,
                    "Comparison": comparison,
                    "Metric": metric_label,
                    "Baseline": baseline_label,
                    "Intervention": intervention_label,
                    "Baseline value": fmt_mean_sd(
                        paired["baseline_value"]
                    ),
                    "Intervention value": fmt_mean_sd(
                        paired["intervention_value"]
                    ),
                    "p_raw": raw_p,
                    "n_pairs": int(
                        paired["seed"].nunique()
                    ),
                }
            )


compas_race_binary_significance_detailed = pd.DataFrame(
    rows
)

compas_race_binary_significance_detailed[
    "p_holm"
] = holm_adjust(
    compas_race_binary_significance_detailed[
        "p_raw"
    ]
)

compas_race_binary_significance_detailed[
    "Holm p"
] = compas_race_binary_significance_detailed[
    "p_holm"
].map(
    format_p_value
)

compas_race_binary_significance_detailed[
    "Sig."
] = compas_race_binary_significance_detailed[
    "p_holm"
].map(
    format_significance
)

compas_race_binary_significance = compas_race_binary_significance_detailed[
    [
        "Family",
        "Comparison",
        "Metric",
        "Baseline",
        "Intervention",
        "Baseline value",
        "Intervention value",
        "Holm p",
        "Sig.",
    ]
].copy()

display(
    compas_race_binary_significance
)


detailed_csv_path = (
    TABLE_DIR
    / "compas_race_binary_significance_tests_detailed.csv"
)

compact_csv_path = (
    TABLE_DIR
    / "compas_race_binary_significance_tests.csv"
)

compact_tex_path = (
    TABLE_DIR
    / "compas_race_binary_significance_tests.tex"
)

compas_race_binary_significance_detailed.to_csv(
    detailed_csv_path,
    index=False,
)

compas_race_binary_significance.to_csv(
    compact_csv_path,
    index=False,
)

export_latex_table(
    compas_race_binary_significance,
    compact_tex_path,
    caption=(
        "Holm-corrected paired Wilcoxon signed-rank tests across seeds for the "
        "COMPAS race-binary experiments. Online comparisons are evaluated at "
        "the final online endpoint. Post-processing comparisons are evaluated on "
        "the held-out test set."
    ),
    label="tab:compas_race_binary_significance_tests",
)

print("Saved:", detailed_csv_path)
print("Saved:", compact_csv_path)
print("Saved:", compact_tex_path)

In [ ]:
# ============================================================
# COMPAS: Random Forest vs fairness-aware contextual bandits
# SAME held-out test set
# ============================================================

FAIR_TEST_POLICIES = [
    "FairLinUCB",
    "FairLinTS",
    "FairEXP4",
]

# Reload RF if necessary
if "compas_rf_df" not in globals():
    compas_rf_df = normalize_metric_columns(
        pd.read_csv(
            RUN_DIR / "random_forest_endpoint.csv"
        )
    )


# ------------------------------------------------------------
# The non-+PP Fair* rows of postproc_df correspond to the
# fairness-aware policies evaluated on the held-out test set.
# ------------------------------------------------------------

compas_supervised_comparison_df = pd.concat(
    [
        compas_rf_df[
            compas_rf_df["policy"]
            == "RandomForest"
        ].copy(),

        postproc_df[
            postproc_df["policy"].isin(
                FAIR_TEST_POLICIES
            )
        ].copy(),
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

counts = (
    compas_supervised_comparison_df
    .groupby(
        ["policy", "preprocessing"]
    )["seed"]
    .agg(
        ["size", "nunique"]
    )
)

display(counts)

assert (counts["size"] == 50).all()
assert (counts["nunique"] == 50).all()

assert set(
    compas_supervised_comparison_df[
        "preprocessing"
    ]
) == {
    "uniform",
    "reweigh_group_label",
}


# ------------------------------------------------------------
# Mean and 95% Student-t CI
# ------------------------------------------------------------

METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]

summary_rows = []

for (
    policy,
    preprocessing,
), group in compas_supervised_comparison_df.groupby(
    [
        "policy",
        "preprocessing",
    ]
):

    n = len(group)

    t_crit = t.ppf(
        0.975,
        df=n - 1,
    )

    row = {
        "policy": policy,
        "preprocessing": preprocessing,
        "n": n,
    }

    for metric in METRICS:

        mean = group[metric].mean()

        sd = group[metric].std(
            ddof=1
        )

        ci95 = (
            t_crit
            * sd
            / np.sqrt(n)
        )

        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95"
        ] = ci95

        row[
            f"{metric}_low"
        ] = mean - ci95

        row[
            f"{metric}_high"
        ] = mean + ci95

    summary_rows.append(
        row
    )


compas_supervised_summary_df = pd.DataFrame(
    summary_rows
)


POLICY_ORDER = [
    "RandomForest",
    "FairLinUCB",
    "FairLinTS",
    "FairEXP4",
]

PREP_ORDER = [
    "uniform",
    "reweigh_group_label",
]


compas_supervised_summary_df[
    "policy"
] = pd.Categorical(
    compas_supervised_summary_df[
        "policy"
    ],
    categories=POLICY_ORDER,
    ordered=True,
)

compas_supervised_summary_df[
    "preprocessing"
] = pd.Categorical(
    compas_supervised_summary_df[
        "preprocessing"
    ],
    categories=PREP_ORDER,
    ordered=True,
)


compas_supervised_summary_df = (
    compas_supervised_summary_df
    .sort_values(
        [
            "policy",
            "preprocessing",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    compas_supervised_summary_df[
        [
            "policy",
            "preprocessing",
            "average_reward_mean",
            "average_reward_ci95",
            "DP_gap_mean",
            "DP_gap_ci95",
            "EO_gap_mean",
            "EO_gap_ci95",
        ]
    ].round(4)
)


SUPERVISED_TABLE_PATH = (
    TABLE_DIR
    / "compas_rf_vs_fair_bandits_summary.csv"
)

compas_supervised_summary_df.to_csv(
    SUPERVISED_TABLE_PATH,
    index=False,
)

print(
    "Saved:",
    SUPERVISED_TABLE_PATH,
)

## Generated artifacts

This final section lists the figures, LaTeX/CSV tables, and raw cached outputs generated by the notebook.

In [ ]:
print("FIGURES")
for path in sorted(FIG_DIR.glob("*.png")):
    print(path)

print("\nTABLES")
for path in sorted(TABLE_DIR.glob("*")):
    print(path)

print("\nRAW OUTPUTS")
for path in sorted(RUN_DIR.rglob("*.csv")):
    print(path)

In [ ]:
# ============================================================
# COMPAS: build/load RF vs fairness-aware bandits summary
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import t

# Paths
RUN_DIR = Path("results/full/compas_race_binary_fairness/full")
FIG_DIR = Path("results/full/compas_race_binary_fairness/final_figures")
TABLE_DIR = Path("results/full/compas_race_binary_fairness/overleaf_tables")

SUMMARY_PATH = TABLE_DIR / "compas_rf_vs_fair_bandits_summary.csv"

# ------------------------------------------------------------
# If summary already exists, simply reload it
# ------------------------------------------------------------

if SUMMARY_PATH.exists():

    print("Loading existing summary:")
    print(SUMMARY_PATH)

    compas_supervised_summary_df = pd.read_csv(
        SUMMARY_PATH
    )

else:

    print("Summary not found: rebuilding from saved CSV files.")

    # Random Forest
    compas_rf_df = pd.read_csv(
        RUN_DIR / "random_forest_endpoint.csv"
    )

    # Bandit held-out test-set results
    postproc_df = pd.concat(
        [
            pd.read_csv(
                RUN_DIR / "linucb" / "postprocessing.csv"
            ),
            pd.read_csv(
                RUN_DIR / "linear_ts" / "postprocessing.csv"
            ),
            pd.read_csv(
                RUN_DIR / "exp4" / "postprocessing.csv"
            ),
        ],
        ignore_index=True,
    )

    FAIR_TEST_POLICIES = [
        "FairLinUCB",
        "FairLinTS",
        "FairEXP4",
    ]

    compas_supervised_comparison_df = pd.concat(
        [
            compas_rf_df[
                compas_rf_df["policy"] == "RandomForest"
            ].copy(),

            postproc_df[
                postproc_df["policy"].isin(
                    FAIR_TEST_POLICIES
                )
            ].copy(),
        ],
        ignore_index=True,
    )

    # Check 50 seeds in every condition
    counts = (
        compas_supervised_comparison_df
        .groupby(
            ["policy", "preprocessing"]
        )["seed"]
        .agg(["size", "nunique"])
    )

    display(counts)

    assert (counts["size"] == 50).all()
    assert (counts["nunique"] == 50).all()

    # --------------------------------------------------------
    # Mean + Student 95% CI
    # --------------------------------------------------------

    METRICS = [
        "average_reward",
        "DP_gap",
        "EO_gap",
    ]

    summary_rows = []

    for (
        policy,
        preprocessing,
    ), group in compas_supervised_comparison_df.groupby(
        ["policy", "preprocessing"]
    ):

        n = len(group)

        t_crit = t.ppf(
            0.975,
            df=n - 1,
        )

        row = {
            "policy": policy,
            "preprocessing": preprocessing,
            "n": n,
        }

        for metric in METRICS:

            mean = group[metric].mean()
            sd = group[metric].std(ddof=1)

            ci95 = (
                t_crit
                * sd
                / np.sqrt(n)
            )

            row[f"{metric}_mean"] = mean
            row[f"{metric}_ci95"] = ci95
            row[f"{metric}_low"] = mean - ci95
            row[f"{metric}_high"] = mean + ci95

        summary_rows.append(row)

    compas_supervised_summary_df = pd.DataFrame(
        summary_rows
    )

    # Save
    TABLE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    compas_supervised_summary_df.to_csv(
        SUMMARY_PATH,
        index=False,
    )

    print(
        "Saved:",
        SUMMARY_PATH,
    )


print("\nCOMPAS supervised comparison:")
display(
    compas_supervised_summary_df[
        [
            "policy",
            "preprocessing",
            "average_reward_mean",
            "average_reward_ci95",
            "DP_gap_mean",
            "DP_gap_ci95",
            "EO_gap_mean",
            "EO_gap_ci95",
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# COMPAS RF comparison — restore variables needed for
# figures and statistical tests
# No model is retrained here.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paths
RUN_DIR = Path("results/full/compas_race_binary_fairness/full")
FIG_DIR = Path("results/full/compas_race_binary_fairness/final_figures")
TABLE_DIR = Path("results/full/compas_race_binary_fairness/overleaf_tables")

# Fixed ordering
PREP_ORDER = [
    "uniform",
    "reweigh_group_label",
]

FAIR_TEST_POLICIES = [
    "FairLinUCB",
    "FairLinTS",
    "FairEXP4",
]

# ------------------------------------------------------------
# Load summary used for the figures
# ------------------------------------------------------------

SUMMARY_PATH = (
    TABLE_DIR
    / "compas_rf_vs_fair_bandits_summary.csv"
)

compas_supervised_summary_df = pd.read_csv(
    SUMMARY_PATH
)

print("Summary loaded:")
display(compas_supervised_summary_df)


# ------------------------------------------------------------
# Rebuild per-seed dataframe needed for Wilcoxon tests
# ------------------------------------------------------------

compas_rf_df = pd.read_csv(
    RUN_DIR / "random_forest_endpoint.csv"
)

postproc_df = pd.concat(
    [
        pd.read_csv(
            RUN_DIR / "linucb" / "postprocessing.csv"
        ),
        pd.read_csv(
            RUN_DIR / "linear_ts" / "postprocessing.csv"
        ),
        pd.read_csv(
            RUN_DIR / "exp4" / "postprocessing.csv"
        ),
    ],
    ignore_index=True,
)

compas_supervised_comparison_df = pd.concat(
    [
        compas_rf_df[
            compas_rf_df["policy"] == "RandomForest"
        ].copy(),

        postproc_df[
            postproc_df["policy"].isin(
                FAIR_TEST_POLICIES
            )
        ].copy(),
    ],
    ignore_index=True,
)

# Safety check
counts = (
    compas_supervised_comparison_df
    .groupby(["policy", "preprocessing"])["seed"]
    .agg(["size", "nunique"])
)

print("\nPer-condition seed counts:")
display(counts)

assert (counts["size"] == 50).all()
assert (counts["nunique"] == 50).all()

print("\nEverything required for figures and statistics is ready.")

In [ ]:
# ============================================================
# COMPAS RF trade-off figures
# ============================================================

COMPARISON_FIG_DIR = (
    FIG_DIR
    / "supervised_baseline"
)

COMPARISON_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SHORT_LABELS = {
    ("RandomForest", "uniform"):
        "RF-U",

    ("RandomForest", "reweigh_group_label"):
        "RF-RW",

    ("FairLinUCB", "uniform"):
        "FairLinUCB-U",

    ("FairLinUCB", "reweigh_group_label"):
        "FairLinUCB-RW",

    ("FairLinTS", "uniform"):
        "FairLinTS-U",

    ("FairLinTS", "reweigh_group_label"):
        "FairLinTS-RW",

    ("FairEXP4", "uniform"):
        "FairEXP4-U",

    ("FairEXP4", "reweigh_group_label"):
        "FairEXP4-RW",
}


def plot_compas_supervised_tradeoff(
    summary_df,
    fairness_metric,
    fairness_label,
    filename,
):

    fig, ax = plt.subplots(
        figsize=(8.2, 5.8)
    )

    for _, row in summary_df.iterrows():

        policy = str(
            row["policy"]
        )

        preprocessing = str(
            row["preprocessing"]
        )

        label = SHORT_LABELS[
            (
                policy,
                preprocessing,
            )
        ]

        marker = (
            "o"
            if preprocessing == "uniform"
            else "s"
        )

        ax.errorbar(
            row[
                f"{fairness_metric}_mean"
            ],
            row[
                "average_reward_mean"
            ],
            xerr=row[
                f"{fairness_metric}_ci95"
            ],
            yerr=row[
                "average_reward_ci95"
            ],
            fmt=marker,
            markersize=6,
            capsize=3,
        )

        ax.annotate(
            label,
            (
                row[
                    f"{fairness_metric}_mean"
                ],
                row[
                    "average_reward_mean"
                ],
            ),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=8.5,
        )

    ax.set_xlabel(
        f"{fairness_label} "
        "(lower is better)"
    )

    ax.set_ylabel(
        "Average reward / accuracy "
        "(higher is better)"
    )

    ax.set_title(
        "COMPAS (race): supervised baseline vs "
        "fairness-aware contextual bandits"
    )

    ax.grid(
        True,
        alpha=0.22,
    )

    fig.tight_layout()

    path = (
        COMPARISON_FIG_DIR
        / filename
    )

    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Saved:",
        path,
    )

    return path


compas_rf_dp_path = (
    plot_compas_supervised_tradeoff(
        compas_supervised_summary_df,
        fairness_metric="DP_gap",
        fairness_label=(
            "Demographic parity gap"
        ),
        filename=(
            "compas_rf_vs_fair_bandits_dp.png"
        ),
    )
)


compas_rf_eo_path = (
    plot_compas_supervised_tradeoff(
        compas_supervised_summary_df,
        fairness_metric="EO_gap",
        fairness_label=(
            "Equalized odds gap"
        ),
        filename=(
            "compas_rf_vs_fair_bandits_eo.png"
        ),
    )
)

In [ ]:
# ============================================================
# Restore statistical helper functions
# ============================================================

import numpy as np
from scipy.stats import wilcoxon


def safe_wilcoxon(
    baseline,
    intervention,
) -> float:

    baseline = np.asarray(
        baseline,
        dtype=float,
    )

    intervention = np.asarray(
        intervention,
        dtype=float,
    )

    mask = (
        np.isfinite(baseline)
        & np.isfinite(intervention)
    )

    baseline = baseline[mask]
    intervention = intervention[mask]

    if len(baseline) == 0:
        return np.nan

    if np.allclose(
        baseline,
        intervention,
    ):
        return 1.0

    return float(
        wilcoxon(
            baseline,
            intervention,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto",
        ).pvalue
    )


def holm_adjust(
    p_values,
) -> np.ndarray:

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    p_values = np.where(
        np.isfinite(p_values),
        p_values,
        1.0,
    )

    order = np.argsort(p_values)

    adjusted = np.empty(
        len(p_values),
        dtype=float,
    )

    previous = 0.0

    for rank, index in enumerate(order):

        value = (
            len(p_values)
            - rank
        ) * p_values[index]

        value = min(
            max(
                value,
                previous,
            ),
            1.0,
        )

        adjusted[index] = value
        previous = value

    return adjusted


print("safe_wilcoxon and holm_adjust are ready.")

In [ ]:
# ============================================================
# RF vs fairness-aware policies:
# paired Wilcoxon + Holm
# ============================================================

rf_comparison_rows = []


for preprocessing in PREP_ORDER:

    rf_seed = (
        compas_supervised_comparison_df[
            (
                compas_supervised_comparison_df[
                    "policy"
                ]
                == "RandomForest"
            )
            & (
                compas_supervised_comparison_df[
                    "preprocessing"
                ]
                == preprocessing
            )
        ]
        .set_index("seed")
        .sort_index()
    )


    for policy in FAIR_TEST_POLICIES:

        bandit_seed = (
            compas_supervised_comparison_df[
                (
                    compas_supervised_comparison_df[
                        "policy"
                    ]
                    == policy
                )
                & (
                    compas_supervised_comparison_df[
                        "preprocessing"
                    ]
                    == preprocessing
                )
            ]
            .set_index("seed")
            .sort_index()
        )


        common_seeds = (
            rf_seed.index.intersection(
                bandit_seed.index
            )
        )

        assert len(
            common_seeds
        ) == 50


        for metric in [
            "average_reward",
            "DP_gap",
            "EO_gap",
        ]:

            rf_values = (
                rf_seed.loc[
                    common_seeds,
                    metric,
                ]
                .to_numpy()
            )

            bandit_values = (
                bandit_seed.loc[
                    common_seeds,
                    metric,
                ]
                .to_numpy()
            )

            p_value = safe_wilcoxon(
                rf_values,
                bandit_values,
            )

            rf_comparison_rows.append(
                {
                    "preprocessing":
                        preprocessing,

                    "comparison":
                        (
                            "RandomForest "
                            f"vs {policy}"
                        ),

                    "metric":
                        metric,

                    "RF_mean":
                        rf_values.mean(),

                    "bandit_mean":
                        bandit_values.mean(),

                    "mean_difference_RF_minus_bandit":
                        (
                            rf_values
                            - bandit_values
                        ).mean(),

                    "p_raw":
                        p_value,
                }
            )


compas_rf_significance_df = pd.DataFrame(
    rf_comparison_rows
)


compas_rf_significance_df[
    "p_holm"
] = holm_adjust(
    compas_rf_significance_df[
        "p_raw"
    ]
)


compas_rf_significance_df[
    "significant_0.05"
] = (
    compas_rf_significance_df[
        "p_holm"
    ]
    < 0.05
)


display(
    compas_rf_significance_df
)


RF_SIGNIFICANCE_PATH = (
    TABLE_DIR
    / "compas_rf_vs_fair_bandits_wilcoxon_holm.csv"
)


compas_rf_significance_df.to_csv(
    RF_SIGNIFICANCE_PATH,
    index=False,
)


print(
    "Saved:",
    RF_SIGNIFICANCE_PATH,
)

In [ ]:
# ============================================================
# FINAL 4-PANEL FIGURE
# Adult DP / Adult EO / COMPAS DP / COMPAS EO
# Supervised RF baseline vs fairness-aware contextual bandits
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Load the already-generated summary tables
# ------------------------------------------------------------

ADULT_SUMMARY_PATH = Path(
    "results/full/adult_sex_cmab/overleaf_tables/"
    "adult_rf_vs_fair_bandits_summary.csv"
)

COMPAS_SUMMARY_PATH = Path(
    "results/full/compas_race_binary_fairness/overleaf_tables/"
    "compas_rf_vs_fair_bandits_summary.csv"
)

adult = pd.read_csv(ADULT_SUMMARY_PATH)
compas = pd.read_csv(COMPAS_SUMMARY_PATH)

# Adult summary may use accuracy_* instead of average_reward_*.
if (
    "accuracy_mean" in adult.columns
    and "average_reward_mean" not in adult.columns
):
    adult["average_reward_mean"] = adult["accuracy_mean"]
    adult["average_reward_ci95"] = adult["accuracy_ci95"]

# ------------------------------------------------------------
# 2. Common ordering and labels
# ------------------------------------------------------------

POLICY_ORDER = [
    "RandomForest",
    "FairEXP4",
    "FairLinTS",
    "FairLinUCB",
]

PREP_ORDER = [
    "uniform",
    "reweigh_group_label",
]

SHORT_LABELS = {
    ("RandomForest", "uniform"): "RF-U",
    ("RandomForest", "reweigh_group_label"): "RF-RW",
    ("FairEXP4", "uniform"): "FairEXP4-U",
    ("FairEXP4", "reweigh_group_label"): "FairEXP4-RW",
    ("FairLinTS", "uniform"): "FairLinTS-U",
    ("FairLinTS", "reweigh_group_label"): "FairLinTS-RW",
    ("FairLinUCB", "uniform"): "FairLinUCB-U",
    ("FairLinUCB", "reweigh_group_label"): "FairLinUCB-RW",
}


def order_summary(df):
    df = df.copy()

    df["policy"] = pd.Categorical(
        df["policy"],
        categories=POLICY_ORDER,
        ordered=True,
    )

    df["preprocessing"] = pd.Categorical(
        df["preprocessing"],
        categories=PREP_ORDER,
        ordered=True,
    )

    return (
        df
        .sort_values(
            ["policy", "preprocessing"]
        )
        .reset_index(drop=True)
    )


adult = order_summary(adult)
compas = order_summary(compas)


# ------------------------------------------------------------
# 3. Small label offsets for readability
# ------------------------------------------------------------

OFFSETS = {
    ("Adult", "DP_gap", "RF-U"): (6, 6),
    ("Adult", "DP_gap", "RF-RW"): (6, 6),
    ("Adult", "DP_gap", "FairEXP4-U"): (-72, 7),
    ("Adult", "DP_gap", "FairEXP4-RW"): (6, 7),
    ("Adult", "DP_gap", "FairLinTS-U"): (6, 7),
    ("Adult", "DP_gap", "FairLinTS-RW"): (-8, -15),
    ("Adult", "DP_gap", "FairLinUCB-U"): (-58, -14),
    ("Adult", "DP_gap", "FairLinUCB-RW"): (6, 6),

    ("Adult", "EO_gap", "RF-U"): (6, 6),
    ("Adult", "EO_gap", "RF-RW"): (6, 6),
    ("Adult", "EO_gap", "FairEXP4-U"): (-58, 7),
    ("Adult", "EO_gap", "FairEXP4-RW"): (6, 7),
    ("Adult", "EO_gap", "FairLinTS-U"): (6, 7),
    ("Adult", "EO_gap", "FairLinTS-RW"): (6, 7),
    ("Adult", "EO_gap", "FairLinUCB-U"): (6, 7),
    ("Adult", "EO_gap", "FairLinUCB-RW"): (6, 7),

    ("COMPAS", "DP_gap", "RF-U"): (-45, 8),
    ("COMPAS", "DP_gap", "RF-RW"): (6, 6),
    ("COMPAS", "DP_gap", "FairEXP4-U"): (-62, 7),
    ("COMPAS", "DP_gap", "FairEXP4-RW"): (6, 7),
    ("COMPAS", "DP_gap", "FairLinTS-U"): (6, 7),
    ("COMPAS", "DP_gap", "FairLinTS-RW"): (6, 7),
    ("COMPAS", "DP_gap", "FairLinUCB-U"): (6, 7),
    ("COMPAS", "DP_gap", "FairLinUCB-RW"): (6, -15),

    ("COMPAS", "EO_gap", "RF-U"): (-46, 8),
    ("COMPAS", "EO_gap", "RF-RW"): (6, 6),
    ("COMPAS", "EO_gap", "FairEXP4-U"): (-62, 7),
    ("COMPAS", "EO_gap", "FairEXP4-RW"): (6, 7),
    ("COMPAS", "EO_gap", "FairLinTS-U"): (6, 7),
    ("COMPAS", "EO_gap", "FairLinTS-RW"): (6, 7),
    ("COMPAS", "EO_gap", "FairLinUCB-U"): (6, 7),
    ("COMPAS", "EO_gap", "FairLinUCB-RW"): (6, -15),
}


# ------------------------------------------------------------
# 4. Four-panel figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.5, 10.2),
)

PANELS = [
    (
        axes[0, 0],
        adult,
        "Adult",
        "DP_gap",
        "Demographic parity gap",
        "A",
    ),
    (
        axes[0, 1],
        adult,
        "Adult",
        "EO_gap",
        "Equalized odds gap",
        "B",
    ),
    (
        axes[1, 0],
        compas,
        "COMPAS",
        "DP_gap",
        "Demographic parity gap",
        "C",
    ),
    (
        axes[1, 1],
        compas,
        "COMPAS",
        "EO_gap",
        "Equalized odds gap",
        "D",
    ),
]


for (
    ax,
    df,
    dataset,
    metric,
    metric_label,
    panel_letter,
) in PANELS:

    for _, row in df.iterrows():

        policy = str(
            row["policy"]
        )

        preprocessing = str(
            row["preprocessing"]
        )

        label = SHORT_LABELS[
            (policy, preprocessing)
        ]

        marker = (
            "o"
            if preprocessing == "uniform"
            else "s"
        )

        ax.errorbar(
            row[f"{metric}_mean"],
            row["average_reward_mean"],
            xerr=row[f"{metric}_ci95"],
            yerr=row["average_reward_ci95"],
            fmt=marker,
            markersize=6,
            capsize=3,
        )

        dx, dy = OFFSETS.get(
            (
                dataset,
                metric,
                label,
            ),
            (6, 6),
        )

        ax.annotate(
            label,
            (
                row[f"{metric}_mean"],
                row["average_reward_mean"],
            ),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=8.3,
        )

    ax.set_xlabel(
        f"{metric_label} "
        "(lower is better)"
    )

    ax.set_ylabel(
        "Average reward / accuracy "
        "(higher is better)"
    )

    ax.set_title(
        f"{panel_letter}. {dataset}"
    )

    ax.grid(
        True,
        alpha=0.22,
    )


fig.suptitle(
    "Supervised Random Forest baseline versus "
    "fairness-aware contextual bandits",
    fontsize=14,
    y=0.995,
)

fig.text(
    0.5,
    0.01,
    (
        "Held-out test-set evaluation; points are means across 50 seeds "
        "and error bars are 95% confidence intervals. "
        "U = uniform preprocessing; RW = group–label reweighting."
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout(
    rect=[
        0,
        0.04,
        1,
        0.97,
    ]
)


# ------------------------------------------------------------
# 5. Save article-ready outputs
# ------------------------------------------------------------

FINAL_FIG_DIR = Path(
    "results/full/comparative_figures"
)

FINAL_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PNG_PATH = (
    FINAL_FIG_DIR
    / "adult_compas_rf_vs_fair_bandits_4panel.png"
)

SVG_PATH = (
    FINAL_FIG_DIR
    / "adult_compas_rf_vs_fair_bandits_4panel.svg"
)

fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    SVG_PATH,
    bbox_inches="tight",
)

plt.show()

print("Saved PNG:", PNG_PATH)
print("Saved SVG:", SVG_PATH)


In [ ]:
# ============================================================
# COMPAS ablation — prerequisite check
# ============================================================

required = [
    "RUN_DIR",
    "FIG_DIR",
    "TABLE_DIR",
    "X_train",
    "X_cal",
    "X_test",
    "y_train",
    "y_cal",
    "y_test",
    "g_train",
    "g_cal",
    "g_test",
    "params",
    "SEEDS",
    "PREPROCESSINGS",
]

missing = [
    name
    for name in required
    if name not in globals()
]

print("Missing:", missing)

In [ ]:
# ============================================================
# COMPAS — full ablation on LinUCB backbone
#
# Missing conditions only:
# standard LinUCB held-out evaluation + LinUCB+PP
# ============================================================

import json
import numpy as np
import pandas as pd

from fair_bandits.experiments.adult_runner import (
    run_adult_family_trajectory,
)

from fair_bandits.experiments.adult_scoring import (
    adult_linucb_score_table,
)

from fair_bandits.postprocessing import (
    actions_from_group_thresholds,
    optimize_group_thresholds,
)

from fair_bandits.metrics import (
    normalize_metric_columns,
    summarize_classification_bandit,
)


# ------------------------------------------------------------
# Separate directory: does NOT touch existing COMPAS results
# ------------------------------------------------------------

ABLATION_DIR = RUN_DIR / "ablation"

ABLATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


STANDARD_ABLATION_PATH = (
    ABLATION_DIR
    / "compas_linucb_standard_holdout_postprocessing.csv"
)

STANDARD_ABLATION_PARAMETERS_PATH = (
    ABLATION_DIR
    / "compas_linucb_standard_postprocessing_parameters.csv"
)


ABLATION_FORCE_RERUN = False


# ------------------------------------------------------------
# Reload partial results if already present
# ------------------------------------------------------------

if (
    STANDARD_ABLATION_PATH.exists()
    and not ABLATION_FORCE_RERUN
):

    compas_linucb_standard_ablation_df = pd.read_csv(
        STANDARD_ABLATION_PATH
    )

else:

    compas_linucb_standard_ablation_df = pd.DataFrame()


if (
    STANDARD_ABLATION_PARAMETERS_PATH.exists()
    and not ABLATION_FORCE_RERUN
):

    compas_linucb_standard_ablation_parameters_df = (
        pd.read_csv(
            STANDARD_ABLATION_PARAMETERS_PATH
        )
    )

else:

    compas_linucb_standard_ablation_parameters_df = (
        pd.DataFrame()
    )


# ------------------------------------------------------------
# Detect completed seed/preprocessing pairs
# ------------------------------------------------------------

done = set()


if not compas_linucb_standard_ablation_df.empty:

    for (
        seed,
        preprocessing,
    ), group in (
        compas_linucb_standard_ablation_df
        .groupby(
            ["seed", "preprocessing"]
        )
    ):

        policies_present = set(
            group["policy"].astype(str)
        )

        if {
            "LinUCB",
            "LinUCB+PP",
        }.issubset(
            policies_present
        ):

            done.add(
                (
                    int(seed),
                    str(preprocessing),
                )
            )


total = (
    len(SEEDS)
    * len(PREPROCESSINGS)
)

completed = len(done)


print(
    f"Already complete: "
    f"{completed}/{total}"
)


# ------------------------------------------------------------
# Run only missing combinations
# ------------------------------------------------------------

for seed in SEEDS:

    for preprocessing in PREPROCESSINGS:

        key = (
            int(seed),
            str(preprocessing),
        )


        if key in done:

            print(
                "Cached ablation:",
                key,
            )

            continue


        print(
            "Running COMPAS ablation:",
            key,
        )


        # ----------------------------------------------------
        # Train STANDARD LinUCB
        # ----------------------------------------------------

        _, linucb_policy = (
            run_adult_family_trajectory(

                family="linucb",

                X_train=X_train,
                y_train=y_train,
                g_train=g_train,

                advice_train=None,

                seed=int(seed),

                policy_name="LinUCB",

                preprocessing=str(
                    preprocessing
                ),

                params=params,
            )
        )


        # ----------------------------------------------------
        # Frozen policy -> calibration
        # ----------------------------------------------------

        calibration_table = (
            adult_linucb_score_table(

                linucb_policy,

                X_cal,
                y_cal,
                g_cal,

                fair=False,
            )
        )


        # ----------------------------------------------------
        # Frozen policy -> test
        # ----------------------------------------------------

        test_table = (
            adult_linucb_score_table(

                linucb_policy,

                X_test,
                y_test,
                g_test,

                fair=False,
            )
        )


        # ----------------------------------------------------
        # Optimize group thresholds on CALIBRATION ONLY
        # ----------------------------------------------------

        (
            thresholds,
            calibration_raw,
            calibration_postprocessed,
        ) = optimize_group_thresholds(

            calibration_table,

            metric_fn=
                summarize_classification_bandit,

            max_accuracy_drop=
                params.max_accuracy_drop,

            threshold_grid_size=
                params.threshold_grid_size,
        )


        # ----------------------------------------------------
        # Threshold zero = original LinUCB
        # ----------------------------------------------------

        zero_thresholds = {

            str(group): 0.0

            for group in sorted(
                np.unique(
                    np.asarray(
                        g_test
                    ).astype(str)
                )
            )
        }


        raw_actions = (
            actions_from_group_thresholds(
                test_table,
                zero_thresholds,
            )
        )


        postprocessed_actions = (
            actions_from_group_thresholds(
                test_table,
                thresholds,
            )
        )


        # ----------------------------------------------------
        # Remove incomplete previous result for same key
        # ----------------------------------------------------

        if not compas_linucb_standard_ablation_df.empty:

            keep = ~(
                (
                    compas_linucb_standard_ablation_df[
                        "seed"
                    ].astype(int)
                    == int(seed)
                )
                &
                (
                    compas_linucb_standard_ablation_df[
                        "preprocessing"
                    ].astype(str)
                    == str(preprocessing)
                )
            )

            compas_linucb_standard_ablation_df = (
                compas_linucb_standard_ablation_df
                .loc[keep]
                .copy()
            )


        # ----------------------------------------------------
        # Held-out metrics
        # ----------------------------------------------------

        new_rows = []


        for policy_name, actions in [

            (
                "LinUCB",
                raw_actions,
            ),

            (
                "LinUCB+PP",
                postprocessed_actions,
            ),

        ]:

            new_rows.append(
                {
                    "family":
                        "linucb",

                    "seed":
                        int(seed),

                    "preprocessing":
                        str(preprocessing),

                    "policy":
                        policy_name,

                    "t":
                        int(params.t_max),

                    **summarize_classification_bandit(
                        y_test,
                        actions,
                        g_test,
                    ),
                }
            )


        compas_linucb_standard_ablation_df = (
            pd.concat(
                [
                    compas_linucb_standard_ablation_df,
                    pd.DataFrame(
                        new_rows
                    ),
                ],
                ignore_index=True,
            )
        )


        # ----------------------------------------------------
        # Save threshold information
        # ----------------------------------------------------

        parameter_row = {

            "family":
                "linucb",

            "seed":
                int(seed),

            "preprocessing":
                str(preprocessing),

            "thresholds_json":
                json.dumps(
                    thresholds,
                    sort_keys=True,
                ),

            "calibration_raw_DP_gap":
                calibration_raw[
                    "DP_gap"
                ],

            "calibration_postproc_DP_gap":
                calibration_postprocessed[
                    "DP_gap"
                ],

            "calibration_raw_EO_gap":
                calibration_raw[
                    "EO_gap"
                ],

            "calibration_postproc_EO_gap":
                calibration_postprocessed[
                    "EO_gap"
                ],

            "calibration_raw_accuracy":
                calibration_raw[
                    "accuracy"
                ],

            "calibration_postproc_accuracy":
                calibration_postprocessed[
                    "accuracy"
                ],
        }


        if not (
            compas_linucb_standard_ablation_parameters_df.empty
        ):

            keep = ~(
                (
                    compas_linucb_standard_ablation_parameters_df[
                        "seed"
                    ].astype(int)
                    == int(seed)
                )
                &
                (
                    compas_linucb_standard_ablation_parameters_df[
                        "preprocessing"
                    ].astype(str)
                    == str(preprocessing)
                )
            )

            compas_linucb_standard_ablation_parameters_df = (
                compas_linucb_standard_ablation_parameters_df
                .loc[keep]
                .copy()
            )


        compas_linucb_standard_ablation_parameters_df = (
            pd.concat(
                [
                    compas_linucb_standard_ablation_parameters_df,
                    pd.DataFrame(
                        [parameter_row]
                    ),
                ],
                ignore_index=True,
            )
        )


        # ----------------------------------------------------
        # Save after EVERY completed condition
        # ----------------------------------------------------

        compas_linucb_standard_ablation_df.to_csv(
            STANDARD_ABLATION_PATH,
            index=False,
        )

        compas_linucb_standard_ablation_parameters_df.to_csv(
            STANDARD_ABLATION_PARAMETERS_PATH,
            index=False,
        )


        completed += 1
        done.add(key)

        print(
            f"Completed COMPAS ablation "
            f"{completed}/{total}"
        )


compas_linucb_standard_ablation_df = (
    normalize_metric_columns(
        compas_linucb_standard_ablation_df
    )
)


print(
    "\nStandard LinUCB held-out COMPAS results:"
)


display(
    compas_linucb_standard_ablation_df
    .groupby(
        [
            "policy",
            "preprocessing",
        ]
    )[
        [
            "average_reward",
            "DP_gap",
            "EO_gap",
        ]
    ]
    .agg(
        [
            "mean",
            "std",
        ]
    )
)

In [ ]:
# ============================================================
# Reload existing COMPAS held-out Fair* results
# No model is rerun
# ============================================================

postproc_df = pd.concat(
    [
        pd.read_csv(
            RUN_DIR
            / "linucb"
            / "postprocessing.csv"
        ),

        pd.read_csv(
            RUN_DIR
            / "linear_ts"
            / "postprocessing.csv"
        ),

        pd.read_csv(
            RUN_DIR
            / "exp4"
            / "postprocessing.csv"
        ),
    ],
    ignore_index=True,
)


postproc_df = normalize_metric_columns(
    postproc_df
)


print(
    "postproc_df:",
    postproc_df.shape
)

print(
    postproc_df[
        "policy"
    ].value_counts()
)

In [ ]:
# ============================================================
# COMPAS — assemble complete 8-condition factorial ablation
# ============================================================

from scipy.stats import t


# ------------------------------------------------------------
# Existing FairLinUCB held-out results
# ------------------------------------------------------------

fair_linucb_holdout_df = (

    postproc_df[

        (
            postproc_df[
                "family"
            ].astype(str)
            == "linucb"
        )

        &

        (
            postproc_df[
                "policy"
            ].astype(str).isin(
                [
                    "FairLinUCB",
                    "FairLinUCB+PP",
                ]
            )
        )
    ]

    .copy()
)


# ------------------------------------------------------------
# New standard LinUCB results
# ------------------------------------------------------------

standard_linucb_holdout_df = (
    compas_linucb_standard_ablation_df
    .copy()
)


compas_ablation_df = pd.concat(
    [
        standard_linucb_holdout_df,
        fair_linucb_holdout_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 2 × 2 × 2 factorial mapping
# ------------------------------------------------------------

CONFIG_MAP = {

    (
        "LinUCB",
        "uniform",
    ):
        "None",

    (
        "LinUCB",
        "reweigh_group_label",
    ):
        "Pre only",

    (
        "FairLinUCB",
        "uniform",
    ):
        "In only",

    (
        "LinUCB+PP",
        "uniform",
    ):
        "Post only",

    (
        "FairLinUCB",
        "reweigh_group_label",
    ):
        "Pre + In",

    (
        "LinUCB+PP",
        "reweigh_group_label",
    ):
        "Pre + Post",

    (
        "FairLinUCB+PP",
        "uniform",
    ):
        "In + Post",

    (
        "FairLinUCB+PP",
        "reweigh_group_label",
    ):
        "Pre + In + Post",
}


compas_ablation_df[
    "configuration"
] = [

    CONFIG_MAP[
        (
            str(policy),
            str(preprocessing),
        )
    ]

    for policy, preprocessing in zip(

        compas_ablation_df[
            "policy"
        ],

        compas_ablation_df[
            "preprocessing"
        ],
    )
]


CONFIG_ORDER = [
    "None",
    "Pre only",
    "In only",
    "Post only",
    "Pre + In",
    "Pre + Post",
    "In + Post",
    "Pre + In + Post",
]


compas_ablation_df[
    "configuration"
] = pd.Categorical(

    compas_ablation_df[
        "configuration"
    ],

    categories=
        CONFIG_ORDER,

    ordered=True,
)


# ------------------------------------------------------------
# Check 50 seeds × 8 conditions
# ------------------------------------------------------------

counts = (

    compas_ablation_df

    .groupby(
        "configuration",
        observed=False,
    )[
        "seed"
    ]

    .agg(
        [
            "size",
            "nunique",
        ]
    )
)


display(
    counts
)


assert (
    counts["size"]
    == 50
).all()


assert (
    counts["nunique"]
    == 50
).all()


# ------------------------------------------------------------
# Mean + Student 95% CI
# ------------------------------------------------------------

ABLATION_METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
    "TPR_gap",
    "FPR_gap",
    "UtilityGap",
]


summary_rows = []


for configuration, group in (

    compas_ablation_df

    .groupby(
        "configuration",
        observed=False,
    )
):

    n = len(group)

    t_crit = t.ppf(
        0.975,
        df=n - 1,
    )


    row = {

        "configuration":
            str(configuration),

        "n":
            n,
    }


    for metric in ABLATION_METRICS:

        mean = (
            group[
                metric
            ].mean()
        )

        sd = (
            group[
                metric
            ].std(
                ddof=1
            )
        )

        ci95 = (
            t_crit
            * sd
            / np.sqrt(n)
        )


        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95"
        ] = ci95

        row[
            f"{metric}_low"
        ] = (
            mean
            - ci95
        )

        row[
            f"{metric}_high"
        ] = (
            mean
            + ci95
        )


    summary_rows.append(
        row
    )


compas_ablation_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


compas_ablation_summary_df[
    "configuration"
] = pd.Categorical(

    compas_ablation_summary_df[
        "configuration"
    ],

    categories=
        CONFIG_ORDER,

    ordered=True,
)


compas_ablation_summary_df = (

    compas_ablation_summary_df

    .sort_values(
        "configuration"
    )

    .reset_index(
        drop=True
    )
)


display(
    compas_ablation_summary_df[
        [
            "configuration",

            "average_reward_mean",
            "average_reward_ci95",

            "DP_gap_mean",
            "DP_gap_ci95",

            "EO_gap_mean",
            "EO_gap_ci95",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# SAVE BOTH summary and seed-level results
# ------------------------------------------------------------

COMPAS_ABLATION_SUMMARY_PATH = (
    TABLE_DIR
    / "compas_linucb_full_ablation_summary.csv"
)


COMPAS_ABLATION_SEEDLEVEL_PATH = (
    TABLE_DIR
    / "compas_linucb_full_ablation_seedlevel.csv"
)


compas_ablation_summary_df.to_csv(
    COMPAS_ABLATION_SUMMARY_PATH,
    index=False,
)


compas_ablation_df.to_csv(
    COMPAS_ABLATION_SEEDLEVEL_PATH,
    index=False,
)


print(
    "Saved summary:",
    COMPAS_ABLATION_SUMMARY_PATH,
)

print(
    "Saved seed-level:",
    COMPAS_ABLATION_SEEDLEVEL_PATH,
)

In [ ]:
# ============================================================
# COMPAS — factorial ablation significance analysis
# 2 × 2 × 2 Pre / In / Post
#
# 12 planned edge contrasts
# Paired Wilcoxon tests across the same 50 seeds
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon


# ------------------------------------------------------------
# Reload seed-level results if necessary
# ------------------------------------------------------------

COMPAS_ABLATION_SEEDLEVEL_PATH = (
    TABLE_DIR
    / "compas_linucb_full_ablation_seedlevel.csv"
)

if "compas_ablation_df" not in globals():

    compas_ablation_df = pd.read_csv(
        COMPAS_ABLATION_SEEDLEVEL_PATH
    )


# ------------------------------------------------------------
# 12 factorial edge contrasts
# ------------------------------------------------------------

CONTRASTS = [

    # Effect of PRE
    {
        "intervention": "Pre",
        "context": "None",
        "before": "None",
        "after": "Pre only",
    },

    {
        "intervention": "Pre",
        "context": "In",
        "before": "In only",
        "after": "Pre + In",
    },

    {
        "intervention": "Pre",
        "context": "Post",
        "before": "Post only",
        "after": "Pre + Post",
    },

    {
        "intervention": "Pre",
        "context": "In + Post",
        "before": "In + Post",
        "after": "Pre + In + Post",
    },


    # Effect of IN
    {
        "intervention": "In",
        "context": "None",
        "before": "None",
        "after": "In only",
    },

    {
        "intervention": "In",
        "context": "Pre",
        "before": "Pre only",
        "after": "Pre + In",
    },

    {
        "intervention": "In",
        "context": "Post",
        "before": "Post only",
        "after": "In + Post",
    },

    {
        "intervention": "In",
        "context": "Pre + Post",
        "before": "Pre + Post",
        "after": "Pre + In + Post",
    },


    # Effect of POST
    {
        "intervention": "Post",
        "context": "None",
        "before": "None",
        "after": "Post only",
    },

    {
        "intervention": "Post",
        "context": "Pre",
        "before": "Pre only",
        "after": "Pre + Post",
    },

    {
        "intervention": "Post",
        "context": "In",
        "before": "In only",
        "after": "In + Post",
    },

    {
        "intervention": "Post",
        "context": "Pre + In",
        "before": "Pre + In",
        "after": "Pre + In + Post",
    },
]


METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]


# ------------------------------------------------------------
# Holm-Bonferroni helper
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    p_values = np.where(
        np.isfinite(p_values),
        p_values,
        1.0,
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty(
        len(p_values),
        dtype=float,
    )

    previous = 0.0

    for rank, index in enumerate(order):

        value = (
            len(p_values)
            - rank
        ) * p_values[index]

        value = min(
            max(
                value,
                previous,
            ),
            1.0,
        )

        adjusted[index] = value
        previous = value

    return adjusted


# ------------------------------------------------------------
# Run paired contrasts
# ------------------------------------------------------------

rows = []


for contrast in CONTRASTS:

    before_df = (
        compas_ablation_df[
            compas_ablation_df[
                "configuration"
            ].astype(str)
            == contrast["before"]
        ]
        .set_index("seed")
        .sort_index()
    )

    after_df = (
        compas_ablation_df[
            compas_ablation_df[
                "configuration"
            ].astype(str)
            == contrast["after"]
        ]
        .set_index("seed")
        .sort_index()
    )


    common_seeds = (
        before_df.index
        .intersection(
            after_df.index
        )
    )

    assert len(common_seeds) == 50


    for metric in METRICS:

        before_values = (
            before_df.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )

        after_values = (
            after_df.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )


        raw_difference = (
            after_values
            - before_values
        )


        # Positive = improvement
        if metric == "average_reward":

            benefit_difference = (
                after_values
                - before_values
            )

        else:

            benefit_difference = (
                before_values
                - after_values
            )


        if np.allclose(
            before_values,
            after_values,
        ):

            statistic = 0.0
            p_value = 1.0

        else:

            result = wilcoxon(
                before_values,
                after_values,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto",
            )

            statistic = result.statistic
            p_value = result.pvalue


        rows.append(
            {
                "intervention":
                    contrast["intervention"],

                "context":
                    contrast["context"],

                "before":
                    contrast["before"],

                "after":
                    contrast["after"],

                "metric":
                    metric,

                "before_mean":
                    before_values.mean(),

                "after_mean":
                    after_values.mean(),

                "raw_difference_after_minus_before":
                    raw_difference.mean(),

                "benefit_difference":
                    benefit_difference.mean(),

                "wilcoxon_statistic":
                    statistic,

                "p_raw":
                    p_value,
            }
        )


compas_ablation_significance_df = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# Global Holm correction across all 36 planned tests
# ------------------------------------------------------------

compas_ablation_significance_df[
    "p_holm_global"
] = holm_adjust(
    compas_ablation_significance_df[
        "p_raw"
    ]
)


compas_ablation_significance_df[
    "significant_global_0.05"
] = (
    compas_ablation_significance_df[
        "p_holm_global"
    ]
    < 0.05
)


# ------------------------------------------------------------
# Optional: Holm within each metric
# ------------------------------------------------------------

compas_ablation_significance_df[
    "p_holm_within_metric"
] = np.nan


for metric in METRICS:

    mask = (
        compas_ablation_significance_df[
            "metric"
        ]
        == metric
    )

    compas_ablation_significance_df.loc[
        mask,
        "p_holm_within_metric",
    ] = holm_adjust(
        compas_ablation_significance_df.loc[
            mask,
            "p_raw",
        ]
    )


compas_ablation_significance_df[
    "significant_within_metric_0.05"
] = (
    compas_ablation_significance_df[
        "p_holm_within_metric"
    ]
    < 0.05
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    compas_ablation_significance_df[
        [
            "intervention",
            "context",
            "before",
            "after",
            "metric",
            "before_mean",
            "after_mean",
            "benefit_difference",
            "p_raw",
            "p_holm_global",
            "significant_global_0.05",
        ]
    ].round(5)
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

COMPAS_ABLATION_STATS_PATH = (
    TABLE_DIR
    / "compas_linucb_full_ablation_wilcoxon_holm.csv"
)

compas_ablation_significance_df.to_csv(
    COMPAS_ABLATION_STATS_PATH,
    index=False,
)

print(
    "Saved:",
    COMPAS_ABLATION_STATS_PATH,
)

In [ ]:
# ============================================================
# COMPAS — 3-panel factorial ablation figure
#
# A. Change in average reward vs no intervention
# B. Reduction in DP gap vs no intervention
# C. Reduction in EO gap vs no intervention
#
# Positive values ALWAYS indicate improvement.
# 95% CI are based on paired seed-level differences.
# ============================================================


import matplotlib.pyplot as plt

from scipy.stats import t


# ------------------------------------------------------------
# Reload seed-level data if necessary
# ------------------------------------------------------------

COMPAS_ABLATION_SEEDLEVEL_PATH = (
    TABLE_DIR
    / "compas_linucb_full_ablation_seedlevel.csv"
)

if "compas_ablation_df" not in globals():

    compas_ablation_df = pd.read_csv(
        COMPAS_ABLATION_SEEDLEVEL_PATH
    )


CONFIG_ORDER = [
    "Pre only",
    "In only",
    "Post only",
    "Pre + In",
    "Pre + Post",
    "In + Post",
    "Pre + In + Post",
]


# ------------------------------------------------------------
# Baseline
# ------------------------------------------------------------

baseline = (
    compas_ablation_df[
        compas_ablation_df[
            "configuration"
        ].astype(str)
        == "None"
    ]
    .set_index("seed")
    .sort_index()
)

assert len(baseline) == 50


# ------------------------------------------------------------
# Paired differences vs baseline
# ------------------------------------------------------------

delta_rows = []


for configuration in CONFIG_ORDER:

    config_df = (
        compas_ablation_df[
            compas_ablation_df[
                "configuration"
            ].astype(str)
            == configuration
        ]
        .set_index("seed")
        .sort_index()
    )

    common_seeds = (
        baseline.index
        .intersection(
            config_df.index
        )
    )

    assert len(common_seeds) == 50


    for metric in [
        "average_reward",
        "DP_gap",
        "EO_gap",
    ]:

        baseline_values = (
            baseline.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )

        config_values = (
            config_df.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )


        # Positive = improvement
        if metric == "average_reward":

            paired_difference = (
                config_values
                - baseline_values
            )

        else:

            paired_difference = (
                baseline_values
                - config_values
            )


        n = len(
            paired_difference
        )

        mean_difference = (
            paired_difference.mean()
        )

        sd_difference = (
            paired_difference.std(
                ddof=1
            )
        )

        t_crit = t.ppf(
            0.975,
            df=n - 1,
        )

        ci95 = (
            t_crit
            * sd_difference
            / np.sqrt(n)
        )


        delta_rows.append(
            {
                "configuration":
                    configuration,

                "metric":
                    metric,

                "mean_difference":
                    mean_difference,

                "ci95":
                    ci95,

                "ci_low":
                    mean_difference
                    - ci95,

                "ci_high":
                    mean_difference
                    + ci95,

                "n":
                    n,
            }
        )


compas_ablation_delta_df = pd.DataFrame(
    delta_rows
)


# ------------------------------------------------------------
# Display values used in figure
# ------------------------------------------------------------

display(
    compas_ablation_delta_df
    .pivot(
        index="configuration",
        columns="metric",
        values="mean_difference",
    )
    .loc[
        CONFIG_ORDER
    ]
    .round(4)
)


# ------------------------------------------------------------
# 3-panel figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 6.2),
    sharey=True,
)


PANEL_INFO = [

    (
        "average_reward",
        "A. Predictive utility",
        "Change in average reward",
    ),

    (
        "DP_gap",
        "B. Demographic parity",
        "Reduction in DP gap",
    ),

    (
        "EO_gap",
        "C. Equalized odds",
        "Reduction in EO gap",
    ),
]


y_positions = np.arange(
    len(CONFIG_ORDER)
)


for ax, (
    metric,
    title,
    xlabel,
) in zip(
    axes,
    PANEL_INFO,
):

    panel_df = (
        compas_ablation_delta_df[
            compas_ablation_delta_df[
                "metric"
            ]
            == metric
        ]
        .set_index(
            "configuration"
        )
        .loc[
            CONFIG_ORDER
        ]
    )


    ax.errorbar(
        panel_df[
            "mean_difference"
        ].to_numpy(),

        y_positions,

        xerr=panel_df[
            "ci95"
        ].to_numpy(),

        fmt="o",

        markersize=6,

        capsize=3,

        linewidth=1.2,
    )


    ax.axvline(
        0,
        linewidth=1,
        linestyle="--",
    )


    ax.set_title(
        title,
        fontsize=11,
    )

    ax.set_xlabel(
        xlabel,
        fontsize=10,
    )

    ax.grid(
        True,
        axis="x",
        alpha=0.20,
    )


axes[0].set_yticks(
    y_positions
)

axes[0].set_yticklabels(
    CONFIG_ORDER,
    fontsize=9,
)

axes[0].invert_yaxis()


fig.suptitle(
    (
        "COMPAS — factorial ablation of preprocessing, "
        "in-processing and post-processing"
    ),
    fontsize=14,
)


fig.text(
    0.5,
    0.01,
    (
        "Effects are paired differences relative to the no-intervention "
        "configuration across 50 seeds. "
        "Positive values indicate improvement; negative values indicate deterioration. "
        "Error bars represent 95% confidence intervals of the paired differences."
    ),
    ha="center",
    fontsize=9,
)


fig.tight_layout(
    rect=[
        0,
        0.06,
        1,
        0.94,
    ]
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

COMPAS_ABLATION_FIG_DIR = (
    FIG_DIR
    / "ablation"
)

COMPAS_ABLATION_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COMPAS_ABLATION_PNG_PATH = (
    COMPAS_ABLATION_FIG_DIR
    / "compas_linucb_full_ablation_3panel.png"
)

COMPAS_ABLATION_SVG_PATH = (
    COMPAS_ABLATION_FIG_DIR
    / "compas_linucb_full_ablation_3panel.svg"
)


fig.savefig(
    COMPAS_ABLATION_PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    COMPAS_ABLATION_SVG_PATH,
    bbox_inches="tight",
)


plt.show()


print(
    "Saved PNG:",
    COMPAS_ABLATION_PNG_PATH,
)

print(
    "Saved SVG:",
    COMPAS_ABLATION_SVG_PATH,
)

In [ ]:
# ============================================================
# FINAL COMPARATIVE ABLATION FIGURE
# Adult + COMPAS
#
# Row 1: Adult
# Row 2: COMPAS
#
# Columns:
# 1. Predictive utility
# 2. Demographic parity
# 3. Equalized odds
#
# Positive values ALWAYS indicate improvement.
# 95% CI are calculated on paired seed-level differences.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ADULT_SEEDLEVEL_PATH = (
    RESULTS_ROOT
    / "adult_sex_cmab"
    / "overleaf_tables"
    / "adult_linucb_full_ablation_seedlevel.csv"
)

COMPAS_SEEDLEVEL_PATH = (
    RESULTS_ROOT
    / "compas_race_binary_fairness"
    / "overleaf_tables"
    / "compas_linucb_full_ablation_seedlevel.csv"
)


print("Adult:", ADULT_SEEDLEVEL_PATH)
print("COMPAS:", COMPAS_SEEDLEVEL_PATH)

assert ADULT_SEEDLEVEL_PATH.exists()
assert COMPAS_SEEDLEVEL_PATH.exists()


adult_ablation = pd.read_csv(
    ADULT_SEEDLEVEL_PATH,
    keep_default_na=False,
)

compas_ablation = pd.read_csv(
    COMPAS_SEEDLEVEL_PATH,
    keep_default_na=False,
)


# ------------------------------------------------------------
# 2. Configuration order
# ------------------------------------------------------------

CONFIG_ORDER = [
    "Pre only",
    "In only",
    "Post only",
    "Pre + In",
    "Pre + Post",
    "In + Post",
    "Pre + In + Post",
]


METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]


# ------------------------------------------------------------
# 3. Function: paired effect vs no-intervention baseline
#
# Reward:
#     configuration - baseline
#
# DP / EO:
#     baseline - configuration
#
# Therefore:
#     positive = improvement
# ------------------------------------------------------------

def compute_ablation_deltas(
    ablation_df,
    dataset_name,
):

    baseline = (
        ablation_df[
            ablation_df[
                "configuration"
            ].astype(str)
            == "None"
        ]
        .set_index("seed")
        .sort_index()
    )

    assert len(baseline) == 50

    rows = []


    for configuration in CONFIG_ORDER:

        config_df = (
            ablation_df[
                ablation_df[
                    "configuration"
                ].astype(str)
                == configuration
            ]
            .set_index("seed")
            .sort_index()
        )


        common_seeds = (
            baseline.index
            .intersection(
                config_df.index
            )
        )

        assert len(common_seeds) == 50


        for metric in METRICS:

            baseline_values = (
                baseline.loc[
                    common_seeds,
                    metric,
                ]
                .to_numpy(
                    dtype=float
                )
            )

            config_values = (
                config_df.loc[
                    common_seeds,
                    metric,
                ]
                .to_numpy(
                    dtype=float
                )
            )


            # --------------------------------------------
            # Positive ALWAYS means improvement
            # --------------------------------------------

            if metric == "average_reward":

                paired_difference = (
                    config_values
                    - baseline_values
                )

            else:

                paired_difference = (
                    baseline_values
                    - config_values
                )


            n = len(
                paired_difference
            )

            mean_difference = (
                paired_difference.mean()
            )

            sd_difference = (
                paired_difference.std(
                    ddof=1
                )
            )

            t_crit = t.ppf(
                0.975,
                df=n - 1,
            )

            ci95 = (
                t_crit
                * sd_difference
                / np.sqrt(n)
            )


            rows.append(
                {
                    "dataset":
                        dataset_name,

                    "configuration":
                        configuration,

                    "metric":
                        metric,

                    "mean_difference":
                        mean_difference,

                    "ci95":
                        ci95,

                    "ci_low":
                        mean_difference
                        - ci95,

                    "ci_high":
                        mean_difference
                        + ci95,
                }
            )


    return pd.DataFrame(
        rows
    )


# ------------------------------------------------------------
# 4. Compute Adult and COMPAS paired effects
# ------------------------------------------------------------

adult_delta_df = compute_ablation_deltas(
    adult_ablation,
    "Adult",
)

compas_delta_df = compute_ablation_deltas(
    compas_ablation,
    "COMPAS",
)


combined_delta_df = pd.concat(
    [
        adult_delta_df,
        compas_delta_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 5. Common x-axis limits within each metric
#
# Adult and COMPAS therefore use the SAME scale
# for a given column.
# ------------------------------------------------------------

XLIMS = {}


for metric in METRICS:

    metric_df = (
        combined_delta_df[
            combined_delta_df[
                "metric"
            ]
            == metric
        ]
    )


    lower = (
        metric_df[
            "ci_low"
        ].min()
    )

    upper = (
        metric_df[
            "ci_high"
        ].max()
    )


    # Always include zero
    lower = min(
        lower,
        0.0,
    )

    upper = max(
        upper,
        0.0,
    )


    width = (
        upper
        - lower
    )

    padding = (
        0.08 * width
        if width > 0
        else 0.01
    )


    XLIMS[metric] = (
        lower - padding,
        upper + padding,
    )


print("\nShared x-axis limits:")
print(XLIMS)


# ------------------------------------------------------------
# 6. Create 2 × 3 figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 10.0),
    sharey=True,
)


PANEL_INFO = [

    (
        "average_reward",
        "Predictive utility",
        "Change in average reward",
    ),

    (
        "DP_gap",
        "Demographic parity",
        "Reduction in DP gap",
    ),

    (
        "EO_gap",
        "Equalized odds",
        "Reduction in EO gap",
    ),
]


DATASETS = [

    (
        "Adult",
        adult_delta_df,
    ),

    (
        "COMPAS",
        compas_delta_df,
    ),
]


PANEL_LETTERS = [
    ["A", "B", "C"],
    ["D", "E", "F"],
]


y_positions = np.arange(
    len(CONFIG_ORDER)
)


for row_index, (
    dataset_name,
    dataset_df,
) in enumerate(DATASETS):


    for col_index, (
        metric,
        column_title,
        xlabel,
    ) in enumerate(PANEL_INFO):


        ax = axes[
            row_index,
            col_index
        ]


        panel_df = (
            dataset_df[
                dataset_df[
                    "metric"
                ]
                == metric
            ]
            .set_index(
                "configuration"
            )
            .loc[
                CONFIG_ORDER
            ]
        )


        ax.errorbar(

            panel_df[
                "mean_difference"
            ].to_numpy(),

            y_positions,

            xerr=panel_df[
                "ci95"
            ].to_numpy(),

            fmt="o",

            markersize=6,

            capsize=3,

            linewidth=1.2,
        )


        # No-effect reference
        ax.axvline(
            0,
            linewidth=1,
            linestyle="--",
        )


        # Same x-axis Adult / COMPAS
        ax.set_xlim(
            XLIMS[
                metric
            ]
        )


        panel_letter = (
            PANEL_LETTERS[
                row_index
            ][
                col_index
            ]
        )


        ax.set_title(
            (
                f"{panel_letter}. "
                f"{dataset_name} — "
                f"{column_title}"
            ),
            fontsize=10.5,
        )


        ax.set_xlabel(
            xlabel,
            fontsize=9.5,
        )


        ax.grid(
            True,
            axis="x",
            alpha=0.20,
        )


# ------------------------------------------------------------
# 7. Y labels
# ------------------------------------------------------------

for row_index in range(2):

    axes[
        row_index,
        0
    ].set_yticks(
        y_positions
    )

    axes[
        row_index,
        0
    ].set_yticklabels(
        CONFIG_ORDER,
        fontsize=9,
    )


# Same vertical ordering
axes[0, 0].invert_yaxis()


# ------------------------------------------------------------
# 8. Global title
# ------------------------------------------------------------

fig.suptitle(
    (
        "Factorial ablation of preprocessing, "
        "in-processing and post-processing"
    ),
    fontsize=14,
    y=0.985,
)


# ------------------------------------------------------------
# 9. Explanatory footer
# ------------------------------------------------------------

fig.text(
    0.5,
    0.012,
    (
        "Effects are paired differences relative to the no-intervention "
        "configuration across 50 seeds. Positive values indicate improvement "
        "and negative values deterioration. Error bars represent 95% confidence "
        "intervals of the paired differences. Identical horizontal scales are "
        "used for Adult and COMPAS within each outcome."
    ),
    ha="center",
    fontsize=8.8,
)


fig.tight_layout(
    rect=[
        0,
        0.055,
        1,
        0.955,
    ],
    h_pad=3.0,
)


# ------------------------------------------------------------
# 10. Save publication-ready figure
# ------------------------------------------------------------

COMPARATIVE_FIG_DIR = (
    RESULTS_ROOT
    / "comparative_figures"
)

COMPARATIVE_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COMBINED_PNG_PATH = (
    COMPARATIVE_FIG_DIR
    / "adult_compas_linucb_ablation_2x3.png"
)

COMBINED_SVG_PATH = (
    COMPARATIVE_FIG_DIR
    / "adult_compas_linucb_ablation_2x3.svg"
)


fig.savefig(
    COMBINED_PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    COMBINED_SVG_PATH,
    bbox_inches="tight",
)


plt.show()


print(
    "Saved PNG:",
    COMBINED_PNG_PATH,
)

print(
    "Saved SVG:",
    COMBINED_SVG_PATH,
)

In [ ]:
# ============================================================
# Adult + COMPAS — temporal fairness summaries
#
# 1. Normalized temporal DP burden = normalized area under DP(t)
# 2. Maximum DP gap
#
# No model is retrained.
# ============================================================






# ------------------------------------------------------------
# 1. Load temporal results
# ------------------------------------------------------------

ADULT_RUN_DIR = (
    RESULTS_ROOT
    / "adult_sex_cmab"
    / "full"
)

COMPAS_RUN_DIR = (
    RESULTS_ROOT
    / "compas_race_binary_fairness"
    / "full"
)


FAMILIES = [
    "linucb",
    "linear_ts",
    "exp4",
]


def load_temporal(run_dir, dataset):

    frames = []

    for family in FAMILIES:

        path = (
            run_dir
            / family
            / "temporal.csv"
        )

        df = pd.read_csv(path)

        df["dataset"] = dataset
        df["family"] = family

        frames.append(df)

    return pd.concat(
        frames,
        ignore_index=True,
    )


adult_temporal = load_temporal(
    ADULT_RUN_DIR,
    "Adult",
)

compas_temporal = load_temporal(
    COMPAS_RUN_DIR,
    "COMPAS",
)


all_temporal = pd.concat(
    [
        adult_temporal,
        compas_temporal,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 2. Per-seed temporal summaries
# ------------------------------------------------------------

trajectory_rows = []


for (
    dataset,
    family,
    policy,
    preprocessing,
    seed,
), group in all_temporal.groupby(
    [
        "dataset",
        "family",
        "policy",
        "preprocessing",
        "seed",
    ]
):

    group = (
        group
        .sort_values("t")
        .copy()
    )

    x = group["t"].to_numpy(
        dtype=float
    )

    y = group["DP_gap"].to_numpy(
        dtype=float
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        continue


    # Normalized area under the observed DP trajectory
    temporal_dp_burden = (
    np.trapezoid(
        y,
        x,
    )
    /
    (
        x[-1]
        - x[0]
    )
)


    max_dp_gap = np.max(
        y
    )


    trajectory_rows.append(
        {
            "dataset":
                dataset,

            "family":
                family,

            "policy":
                policy,

            "preprocessing":
                preprocessing,

            "seed":
                int(seed),

            "temporal_DP_burden":
                temporal_dp_burden,

            "max_DP_gap":
                max_dp_gap,
        }
    )


temporal_summary_seed_df = pd.DataFrame(
    trajectory_rows
)


# ------------------------------------------------------------
# 3. Mean + 95% CI across 50 seeds
# ------------------------------------------------------------

summary_rows = []


for (
    dataset,
    family,
    policy,
    preprocessing,
), group in temporal_summary_seed_df.groupby(
    [
        "dataset",
        "family",
        "policy",
        "preprocessing",
    ]
):

    n = len(group)

    row = {
        "dataset": dataset,
        "family": family,
        "policy": policy,
        "preprocessing": preprocessing,
        "n": n,
    }

    for metric in [
        "temporal_DP_burden",
        "max_DP_gap",
    ]:

        mean = group[
            metric
        ].mean()

        sd = group[
            metric
        ].std(
            ddof=1
        )

        ci95 = (
            t.ppf(
                0.975,
                df=n - 1,
            )
            * sd
            / np.sqrt(n)
        )

        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95"
        ] = ci95


    summary_rows.append(row)


temporal_fairness_summary_df = pd.DataFrame(
    summary_rows
)


display(
    temporal_fairness_summary_df
    .sort_values(
        [
            "dataset",
            "family",
            "policy",
            "preprocessing",
        ]
    )
    .round(4)
)


# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------

TEMPORAL_TABLE_PATH = (
    RESULTS_ROOT
    / "comparative_figures"
    / "adult_compas_temporal_fairness_summary.csv"
)

TEMPORAL_TABLE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

temporal_fairness_summary_df.to_csv(
    TEMPORAL_TABLE_PATH,
    index=False,
)

print(
    "Saved:",
    TEMPORAL_TABLE_PATH,
)

In [ ]:
# ============================================================
# COMPAS — pilot sensitivity analysis for lambda_DP
#
# FairLinUCB + uniform preprocessing only
# 6 lambda values × 10 seeds
#
# Uses ONLINE endpoint metrics at T = 30,000.
# Existing benchmarks are NOT modified.
# ============================================================

from dataclasses import replace



# ------------------------------------------------------------
# 1. Sensitivity grid
# ------------------------------------------------------------

LAMBDA_GRID = [
    0.0,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
    8.0,
    16.0,
    32.0,
    64.0,
]

SENSITIVITY_SEEDS = list(
    range(50)
)


# ------------------------------------------------------------
# 2. Separate cache
# ------------------------------------------------------------

LAMBDA_SENSITIVITY_DIR = (
    RUN_DIR
    / "lambda_dp_sensitivity"
)

LAMBDA_SENSITIVITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LAMBDA_RESULTS_PATH = (
    LAMBDA_SENSITIVITY_DIR
    / "compas_fairlinucb_lambda_.dp_50seedssv"
)


# ------------------------------------------------------------
# 3. Reload previous partial results
# ------------------------------------------------------------

if LAMBDA_RESULTS_PATH.exists():

    lambda_results_df = pd.read_csv(
        LAMBDA_RESULTS_PATH
    )

else:

    lambda_results_df = pd.DataFrame()


done = set()

if not lambda_results_df.empty:

    done = set(
        zip(
            lambda_results_df[
                "lambda_dp"
            ].astype(float),

            lambda_results_df[
                "seed"
            ].astype(int),
        )
    )


total = (
    len(LAMBDA_GRID)
    * len(SENSITIVITY_SEEDS)
)

print(
    f"Already complete: "
    f"{len(done)}/{total}"
)


# ------------------------------------------------------------
# 4. Run FairLinUCB sensitivity experiment
# ------------------------------------------------------------

rows = []


for lambda_dp in LAMBDA_GRID:

    sensitivity_params = replace(
        params,
        dp_lambda=float(
            lambda_dp
        ),
    )


    for seed in SENSITIVITY_SEEDS:

        key = (
            float(lambda_dp),
            int(seed),
        )


        if key in done:

            print(
                "Cached:",
                key,
            )

            continue


        print(
            "Running:",
            "lambda =",
            lambda_dp,
            "| seed =",
            seed,
        )


        trajectory_df, _ = (
            run_adult_family_trajectory(

                family="linucb",

                X_train=X_train,
                y_train=y_train,
                g_train=g_train,

                advice_train=None,

                seed=int(seed),

                policy_name=
                    "FairLinUCB",

                preprocessing=
                    "uniform",

                params=
                    sensitivity_params,
            )
        )


        # Last recorded online point
        final_row = (
            trajectory_df
            .sort_values("t")
            .iloc[-1]
        )


        new_row = {

            "dataset":
                "COMPAS",

            "lambda_dp":
                float(lambda_dp),

            "seed":
                int(seed),

            "t":
                int(final_row["t"]),

            "average_reward":
                float(
                    final_row[
                        "average_reward"
                    ]
                ),

            "DP_gap":
                float(
                    final_row[
                        "DP_gap"
                    ]
                ),

            "EO_gap":
                float(
                    final_row[
                        "EO_gap"
                    ]
                ),
        }


        lambda_results_df = pd.concat(
            [
                lambda_results_df,
                pd.DataFrame(
                    [new_row]
                ),
            ],
            ignore_index=True,
        )


        lambda_results_df.to_csv(
            LAMBDA_RESULTS_PATH,
            index=False,
        )


        done.add(key)


print(
    "\nCompleted:",
    len(done),
    "/",
    total,
)

In [ ]:
# ============================================================
# Summary — COMPAS lambda_DP pilot
# ============================================================

summary_rows = []


for lambda_dp, group in (
    lambda_results_df
    .groupby("lambda_dp")
):

    n = len(group)

    row = {
        "lambda_dp":
            float(lambda_dp),

        "n":
            n,
    }


    for metric in [
        "average_reward",
        "DP_gap",
        "EO_gap",
    ]:

        mean = group[
            metric
        ].mean()

        sd = group[
            metric
        ].std(
            ddof=1
        )

        ci95 = (
            t.ppf(
                0.975,
                df=n - 1,
            )
            * sd
            / np.sqrt(n)
        )


        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95"
        ] = ci95


    summary_rows.append(
        row
    )


lambda_summary_df = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        "lambda_dp"
    )
)


display(
    lambda_summary_df.round(4)
)

In [ ]:
print(
    "lambda_results_df exists:",
    "lambda_results_df" in globals()
)

if "lambda_results_df" in globals():

    print(
        "Rows:",
        len(lambda_results_df)
    )

    print(
        lambda_results_df
        .groupby("lambda_dp")["seed"]
        .nunique()
    )

In [ ]:
COMPAS_FINAL_PATH = (
    RESULTS_ROOT
    / "compas_race_binary_fairness"
    / "full"
    / "lambda_dp_sensitivity"
    / "compas_fairlinucb_lambda_dp_50seeds.csv"
)

COMPAS_FINAL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

lambda_results_df.to_csv(
    COMPAS_FINAL_PATH,
    index=False
)

print("Saved:", COMPAS_FINAL_PATH)
print("Rows:", len(lambda_results_df))

In [ ]:
# ============================================================
# FINAL FIGURE — lambda_DP sensitivity
# Adult + COMPAS
#
# Row 1: Adult
# Row 2: COMPAS
#
# Columns:
# A/D = Average reward
# B/E = Demographic parity gap
# C/F = Equalized odds gap
#
# FairLinUCB + uniform preprocessing
# 50 seeds
#
# No model is retrained.
# ============================================================


# ------------------------------------------------------------
# 1. Lambda grid
# ------------------------------------------------------------

LAMBDA_GRID = [
    0.0,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
    8.0,
    16.0,
    32.0,
    64.0,
]

MAIN_LAMBDA = 2.0


# ------------------------------------------------------------
# 2. Paths to raw 50-seed results
#
# COMPAS may still have the original "...pilot.csv" filename
# because we deliberately kept it to preserve the cache.
# ------------------------------------------------------------

ADULT_RAW_PATH = (
    RESULTS_ROOT
    / "adult_sex_cmab"
    / "full"
    / "lambda_dp_sensitivity"
    / "adult_fairlinucb_lambda_dp_50seeds.csv"
)


COMPAS_CANDIDATES = [

    (
        RESULTS_ROOT
        / "compas_race_binary_fairness"
        / "full"
        / "lambda_dp_sensitivity"
        / "compas_fairlinucb_lambda_dp_50seeds.csv"
    ),

    (
        RESULTS_ROOT
        / "compas_race_binary_fairness"
        / "full"
        / "lambda_dp_sensitivity"
        / "compas_fairlinucb_lambda_dp_pilot.csv"
    ),
]


assert ADULT_RAW_PATH.exists(), (
    f"Adult file not found:\n{ADULT_RAW_PATH}"
)


COMPAS_RAW_PATH = next(
    (
        path
        for path in COMPAS_CANDIDATES
        if path.exists()
    ),
    None,
)


assert COMPAS_RAW_PATH is not None, (
    "COMPAS sensitivity file not found.\n"
    + "\n".join(
        str(p)
        for p in COMPAS_CANDIDATES
    )
)


print("Adult file:")
print(ADULT_RAW_PATH)

print("\nCOMPAS file:")
print(COMPAS_RAW_PATH)


# ------------------------------------------------------------
# 3. Load raw results
# ------------------------------------------------------------

adult_raw = pd.read_csv(
    ADULT_RAW_PATH
)

compas_raw = pd.read_csv(
    COMPAS_RAW_PATH
)


print(
    "\nAdult rows:",
    len(adult_raw)
)

print(
    "COMPAS rows:",
    len(compas_raw)
)


# ------------------------------------------------------------
# 4. Check completeness
# ------------------------------------------------------------

def check_sensitivity_results(
    df,
    dataset_name,
):

    counts = (
        df
        .groupby("lambda_dp")
        ["seed"]
        .nunique()
        .reindex(LAMBDA_GRID)
    )

    print(
        f"\n{dataset_name} seeds per lambda:"
    )

    print(counts)


    assert (
        counts.notna().all()
    ), (
        f"{dataset_name}: "
        "one or more lambda values are missing."
    )


    assert (
        counts == 50
    ).all(), (
        f"{dataset_name}: "
        "not all lambda values contain 50 seeds."
    )


check_sensitivity_results(
    adult_raw,
    "Adult",
)

check_sensitivity_results(
    compas_raw,
    "COMPAS",
)


# ------------------------------------------------------------
# 5. Mean + 95% Student-t CI
# ------------------------------------------------------------

METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]


def summarize_lambda_results(
    df,
    dataset_name,
):

    rows = []


    for lambda_dp in LAMBDA_GRID:

        group = df[
            np.isclose(
                df["lambda_dp"].astype(float),
                lambda_dp,
            )
        ]


        n = len(group)

        row = {
            "dataset":
                dataset_name,

            "lambda_dp":
                float(lambda_dp),

            "n":
                n,
        }


        for metric in METRICS:

            values = (
                group[metric]
                .to_numpy(
                    dtype=float
                )
            )


            mean = np.mean(
                values
            )

            sd = np.std(
                values,
                ddof=1,
            )

            ci95 = (
                t.ppf(
                    0.975,
                    df=n - 1,
                )
                * sd
                / np.sqrt(n)
            )


            row[
                f"{metric}_mean"
            ] = mean

            row[
                f"{metric}_ci95"
            ] = ci95


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )


adult_summary = summarize_lambda_results(
    adult_raw,
    "Adult",
)

compas_summary = summarize_lambda_results(
    compas_raw,
    "COMPAS",
)


combined_summary = pd.concat(
    [
        adult_summary,
        compas_summary,
    ],
    ignore_index=True,
)


print("\nAdult summary:")
display(
    adult_summary.round(4)
)

print("\nCOMPAS summary:")
display(
    compas_summary.round(4)
)


# ------------------------------------------------------------
# 6. Figure configuration
# ------------------------------------------------------------

PANEL_INFO = [

    (
        "average_reward",
        "Predictive utility",
        "Average reward",
    ),

    (
        "DP_gap",
        "Demographic parity",
        "DP gap",
    ),

    (
        "EO_gap",
        "Equalized odds",
        "EO gap",
    ),
]


DATASETS = [

    (
        "Adult",
        adult_summary,
    ),

    (
        "COMPAS",
        compas_summary,
    ),
]


PANEL_LETTERS = [
    ["A", "B", "C"],
    ["D", "E", "F"],
]


# ------------------------------------------------------------
# 7. Create 2 × 3 figure
#
# sharey='col':
# Adult and COMPAS use the SAME vertical scale
# within each metric.
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 9.2),
    sharex=True,
    sharey="col",
)


# ------------------------------------------------------------
# 8. Plot each panel
# ------------------------------------------------------------

for row_index, (
    dataset_name,
    summary_df,
) in enumerate(DATASETS):


    for col_index, (
        metric,
        metric_title,
        ylabel,
    ) in enumerate(PANEL_INFO):


        ax = axes[
            row_index,
            col_index
        ]


        mean_col = (
            f"{metric}_mean"
        )

        ci_col = (
            f"{metric}_ci95"
        )


        x = summary_df[
            "lambda_dp"
        ].to_numpy(
            dtype=float
        )

        y = summary_df[
            mean_col
        ].to_numpy(
            dtype=float
        )

        yerr = summary_df[
            ci_col
        ].to_numpy(
            dtype=float
        )


        # Mean ± 95% CI
        ax.errorbar(
            x,
            y,
            yerr=yerr,
            marker="o",
            linestyle="-",
            linewidth=1.5,
            markersize=5.5,
            capsize=3,
        )


        # Main lambda used in experiments
        ax.axvline(
            MAIN_LAMBDA,
            linestyle="--",
            linewidth=1.2,
        )


        # Symmetric logarithmic scale:
        # preserves lambda = 0
        ax.set_xscale(
            "symlog",
            linthresh=0.25,
            linscale=1.0,
            base=2,
        )


        ax.set_xticks(
            LAMBDA_GRID
        )

        ax.set_xticklabels(
            [
                "0",
                "0.25",
                "0.5",
                "1",
                "2",
                "4",
                "8",
                "16",
                "32",
                "64",
            ]
        )


        panel_letter = (
            PANEL_LETTERS[
                row_index
            ][
                col_index
            ]
        )


        ax.set_title(
            (
                f"{panel_letter}. "
                f"{dataset_name} — "
                f"{metric_title}"
            ),
            fontsize=10.5,
        )


        ax.set_ylabel(
            ylabel,
            fontsize=9.5,
        )


        ax.grid(
            True,
            axis="y",
            alpha=0.20,
        )


        # Label lambda = 2 only on upper row
        if row_index == 0:

            ax.text(
                MAIN_LAMBDA,
                0.97,
                r"$\lambda_{DP}=2$",
                transform=ax.get_xaxis_transform(),
                rotation=90,
                va="top",
                ha="right",
                fontsize=8.5,
            )


# ------------------------------------------------------------
# 9. X-axis labels
# ------------------------------------------------------------

for ax in axes[1, :]:

    ax.set_xlabel(
        r"Demographic-parity penalty $\lambda_{DP}$",
        fontsize=9.5,
    )


# ------------------------------------------------------------
# 10. Global title
# ------------------------------------------------------------

fig.suptitle(
    (
        r"Sensitivity to the demographic-parity penalty "
        r"$\lambda_{DP}$"
    ),
    fontsize=14,
    y=0.985,
)


# ------------------------------------------------------------
# 11. Explanatory footer
# ------------------------------------------------------------

fig.text(
    0.5,
    0.014,
    (
        "FairLinUCB with uniform preprocessing. "
        "Points represent means across 50 random seeds and error bars denote "
        "95% confidence intervals. The vertical dashed line indicates "
        r"$\lambda_{DP}=2$, the value used in the main experiments."
    ),
    ha="center",
    fontsize=8.8,
)


fig.tight_layout(
    rect=[
        0,
        0.055,
        1,
        0.955,
    ],
    h_pad=2.6,
    w_pad=2.0,
)


# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

COMPARATIVE_FIG_DIR = (
    RESULTS_ROOT
    / "comparative_figures"
)

COMPARATIVE_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PNG_PATH = (
    COMPARATIVE_FIG_DIR
    / "adult_compas_lambda_dp_sensitivity_2x3.png"
)

SVG_PATH = (
    COMPARATIVE_FIG_DIR
    / "adult_compas_lambda_dp_sensitivity_2x3.svg"
)


fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    SVG_PATH,
    bbox_inches="tight",
)


plt.show()


print(
    "\nSaved PNG:",
    PNG_PATH,
)

print(
    "Saved SVG:",
    SVG_PATH,
)


# ------------------------------------------------------------
# 13. Save combined numerical summary
# ------------------------------------------------------------

SUMMARY_PATH = (
    COMPARATIVE_FIG_DIR
    / "adult_compas_lambda_dp_sensitivity_summary.csv"
)


combined_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)


print(
    "Saved summary:",
    SUMMARY_PATH,
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURES S4–S7
# Complete temporal trajectories for LinTS and EXP4
#
# S4 = Adult  — LinTS / FairLinTS
# S5 = COMPAS — LinTS / FairLinTS
# S6 = Adult  — EXP4 / FairEXP4
# S7 = COMPAS — EXP4 / FairEXP4
#
# NO EXPERIMENT IS RERUN.
# ============================================================



# ------------------------------------------------------------
# 1. Directories
# ------------------------------------------------------------

ADULT_RUN_DIR = (
    RESULTS_ROOT
    / "adult_sex_cmab"
    / "full"
)

COMPAS_RUN_DIR = (
    RESULTS_ROOT
    / "compas_race_binary_fairness"
    / "full"
)

SUPP_DIR = (
    RESULTS_ROOT
    / "comparative_figures"
    / "supplementary"
)

SUPP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Find family-specific temporal.csv
# ------------------------------------------------------------

def find_family_temporal_file(
    run_dir,
    family,
):

    candidates = list(
        run_dir.rglob("temporal.csv")
    )

    family = family.lower()

    if family == "lints":
        keywords = [
            "lints",
            "linear_ts",
            "linear-ts",
            "thompson",
        ]

    elif family == "exp4":
        keywords = [
            "exp4",
        ]

    else:
        raise ValueError(
            f"Unknown family: {family}"
        )


    matches = [
        p
        for p in candidates
        if any(
            k in str(p).lower()
            for k in keywords
        )
    ]


    if len(matches) == 0:
        raise FileNotFoundError(
            f"No {family} temporal.csv found under:\n"
            f"{run_dir}\n\n"
            f"Available temporal files:\n"
            + "\n".join(str(p) for p in candidates)
        )


    print(
        f"{run_dir.name} / {family} ->",
        matches[0],
    )

    return matches[0]


PATHS = {

    ("Adult", "LinTS"):
        find_family_temporal_file(
            ADULT_RUN_DIR,
            "lints",
        ),

    ("COMPAS", "LinTS"):
        find_family_temporal_file(
            COMPAS_RUN_DIR,
            "lints",
        ),

    ("Adult", "EXP4"):
        find_family_temporal_file(
            ADULT_RUN_DIR,
            "exp4",
        ),

    ("COMPAS", "EXP4"):
        find_family_temporal_file(
            COMPAS_RUN_DIR,
            "exp4",
        ),
}


# ------------------------------------------------------------
# 3. Load and harmonize one temporal file
# ------------------------------------------------------------

def prepare_temporal(
    path,
    dataset,
    family,
):

    df = pd.read_csv(path)

    if "normalize_metric_columns" in globals():
        df = normalize_metric_columns(df)


    # Time column
    time_col = next(
        c
        for c in [
            "t",
            "T",
            "round",
            "horizon",
        ]
        if c in df.columns
    )


    # Reward column
    reward_col = next(
        c
        for c in [
            "average_reward",
            "avg_reward",
        ]
        if c in df.columns
    )


    # Fairness columns
    dp_col = next(
        c
        for c in [
            "DP_gap",
            "DP_gap_over_time",
        ]
        if c in df.columns
    )

    eo_col = next(
        c
        for c in [
            "EO_gap",
            "EO_gap_over_time",
        ]
        if c in df.columns
    )


    # ------------------------------------------
    # Policy names
    # ------------------------------------------

    if family == "LinTS":

        wanted = {
            "LinTS",
            "linear_ts",
            "FairLinTS",
            "FairLinTS_DP",
        }

        policy_map = {
            "LinTS": "LinTS",
            "linear_ts": "LinTS",
            "FairLinTS": "FairLinTS",
            "FairLinTS_DP": "FairLinTS",
        }

    elif family == "EXP4":

        wanted = {
            "EXP4",
            "FairEXP4",
            "FairEXP4_DP",
        }

        policy_map = {
            "EXP4": "EXP4",
            "FairEXP4": "FairEXP4",
            "FairEXP4_DP": "FairEXP4",
        }


    df = df[
        df["policy"].isin(wanted)
    ].copy()


    df["policy_display"] = (
        df["policy"]
        .map(policy_map)
    )


    # ------------------------------------------
    # Keep U and RW only
    # ------------------------------------------

    df = df[
        df["preprocessing"].isin(
            [
                "uniform",
                "reweigh_group_label",
            ]
        )
    ].copy()


    return pd.DataFrame(
        {
            "dataset":
                dataset,

            "family":
                family,

            "seed":
                df["seed"],

            "t":
                pd.to_numeric(
                    df[time_col],
                    errors="coerce",
                ),

            "policy":
                df["policy_display"],

            "preprocessing":
                df["preprocessing"],

            "average_reward":
                pd.to_numeric(
                    df[reward_col],
                    errors="coerce",
                ),

            "DP_gap":
                pd.to_numeric(
                    df[dp_col],
                    errors="coerce",
                ),

            "EO_gap":
                pd.to_numeric(
                    df[eo_col],
                    errors="coerce",
                ),
        }
    ).dropna()


# ------------------------------------------------------------
# 4. Load all four experiments
# ------------------------------------------------------------

frames = []

for (
    dataset,
    family
), path in PATHS.items():

    frames.append(
        prepare_temporal(
            path,
            dataset,
            family,
        )
    )


temporal_all = pd.concat(
    frames,
    ignore_index=True,
)


print("\nSeeds per configuration:")

display(
    temporal_all
    .groupby(
        [
            "dataset",
            "family",
            "policy",
            "preprocessing",
        ]
    )["seed"]
    .nunique()
    .to_frame("n_seeds")
)


# ------------------------------------------------------------
# 5. Mean + Student-t 95% CI
# ------------------------------------------------------------

METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]


rows = []


for (
    dataset,
    family,
    policy,
    preprocessing,
    horizon,
), g in temporal_all.groupby(
    [
        "dataset",
        "family",
        "policy",
        "preprocessing",
        "t",
    ]
):

    row = {
        "dataset": dataset,
        "family": family,
        "policy": policy,
        "preprocessing": preprocessing,
        "t": horizon,
    }


    for metric in METRICS:

        values = (
            g[metric]
            .to_numpy(dtype=float)
        )

        n = len(values)

        row[f"{metric}_mean"] = (
            values.mean()
        )

        row[f"{metric}_ci95"] = (
            t.ppf(
                0.975,
                n - 1,
            )
            * values.std(ddof=1)
            / np.sqrt(n)
        )


    rows.append(row)


summary = pd.DataFrame(rows)


# ------------------------------------------------------------
# 6. Visual style
# ------------------------------------------------------------

STYLE = {

    "standard_uniform": {
        "color": "tab:blue",
        "linestyle": "-",
        "label": "Standard-U",
    },

    "standard_rw": {
        "color": "tab:blue",
        "linestyle": ":",
        "label": "Standard-RW",
    },

    "fair_uniform": {
        "color": "tab:red",
        "linestyle": "-",
        "label": "Fair-U",
    },

    "fair_rw": {
        "color": "tab:red",
        "linestyle": ":",
        "label": "Fair-RW",
    },
}


def get_style(
    policy,
    preprocessing,
):

    fair = policy.startswith("Fair")

    rw = (
        preprocessing
        == "reweigh_group_label"
    )


    if fair and rw:
        return STYLE["fair_rw"]

    if fair and not rw:
        return STYLE["fair_uniform"]

    if not fair and rw:
        return STYLE["standard_rw"]

    return STYLE["standard_uniform"]


# ------------------------------------------------------------
# 7. Function to create one supplementary figure
# ------------------------------------------------------------

def make_supp_figure(
    dataset,
    family,
    figure_number,
):

    df = summary[
        (summary["dataset"] == dataset)
        &
        (summary["family"] == family)
    ].copy()


    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14.5, 4.5),
    )


    panels = [

        (
            "average_reward",
            "Predictive utility",
            "Average reward",
        ),

        (
            "DP_gap",
            "Demographic parity",
            "DP gap",
        ),

        (
            "EO_gap",
            "Equalized odds",
            "EO gap",
        ),
    ]


    for idx, (
        metric,
        title,
        ylabel,
    ) in enumerate(panels):

        ax = axes[idx]


        for (
            policy,
            preprocessing
        ), g in df.groupby(
            [
                "policy",
                "preprocessing",
            ]
        ):

            g = g.sort_values("t")

            style = get_style(
                policy,
                preprocessing,
            )


            x = g["t"].to_numpy()

            y = g[
                f"{metric}_mean"
            ].to_numpy()

            ci = g[
                f"{metric}_ci95"
            ].to_numpy()


            ax.plot(
                x,
                y,
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=1.8,
                label=style["label"],
            )


            ax.fill_between(
                x,
                y - ci,
                y + ci,
                color=style["color"],
                alpha=0.10,
                linewidth=0,
            )


        letter = chr(
            ord("A") + idx
        )


        ax.set_title(
            f"{letter}. {title}",
            fontsize=10.5,
        )

        ax.set_xlabel(
            r"Decision round $t$"
        )

        ax.set_ylabel(
            ylabel
        )

        ax.grid(
            axis="y",
            alpha=0.20,
        )


    handles, labels = (
        axes[0]
        .get_legend_handles_labels()
    )

    # ------------------------------------------------------------
    # Fixed legend order
    # ------------------------------------------------------------

    from matplotlib.lines import Line2D

    legend_elements = [

        Line2D(
            [0], [0],
            color="tab:blue",
            linestyle="-",
            linewidth=1.8,
            label="Standard-U",
        ),

        Line2D(
            [0], [0],
            color="tab:blue",
            linestyle=":",
            linewidth=1.8,
            label="Standard-RW",
        ),

        Line2D(
            [0], [0],
            color="tab:red",
            linestyle="-",
            linewidth=1.8,
            label="Fair-U",
        ),

        Line2D(
            [0], [0],
            color="tab:red",
            linestyle=":",
            linewidth=1.8,
            label="Fair-RW",
        ),
    ]


    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 0.025),
        fontsize=9.5,
    )


    # ------------------------------------------------------------
    # More explicit title
    # ------------------------------------------------------------

    family_title = {
        "LinTS": "Linear Thompson Sampling",
        "EXP4": "EXP4",
    }[family]


    dataset_title = {
        "Adult": "Adult Income",
        "COMPAS": "COMPAS",
    }[dataset]


    fig.suptitle(
        (
            f"Temporal fairness and utility trajectories "
            f"for {family_title} on {dataset_title}"
        ),
        fontsize=13.5,
        y=0.98,
    )


    # ------------------------------------------------------------
    # Explanatory footer
    # ------------------------------------------------------------

    fig.text(
        0.5,
        0.085,
        (
            "U = uniform preprocessing; "
            "RW = group–label reweighting. "
            "Lines represent means across 50 random seeds and "
            "shaded bands denote 95% confidence intervals."
        ),
        ha="center",
        fontsize=8.8,
    )


    # More room for footer + legend
    fig.tight_layout(
        rect=[0, 0.15, 1, 0.93],
        w_pad=1.8,
    )


    base_name = (
        f"figure_{figure_number}_"
        f"{dataset.lower()}_"
        f"{family.lower()}_temporal"
    )


    png_path = (
        SUPP_DIR
        / f"{base_name}.png"
    )

    svg_path = (
        SUPP_DIR
        / f"{base_name}.svg"
    )


    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        svg_path,
        bbox_inches="tight",
    )


    plt.show()


    print(
        "Saved:",
        png_path
    )

    print(
        "Saved:",
        svg_path
    )


# ------------------------------------------------------------
# 8. Generate S4–S7
# ------------------------------------------------------------

make_supp_figure(
    dataset="Adult",
    family="LinTS",
    figure_number="S4",
)

make_supp_figure(
    dataset="COMPAS",
    family="LinTS",
    figure_number="S5",
)

make_supp_figure(
    dataset="Adult",
    family="EXP4",
    figure_number="S6",
)

make_supp_figure(
    dataset="COMPAS",
    family="EXP4",
    figure_number="S7",
)